# White Micas and the Al-OH Absorption Center: EMIT and AVIRIS-5

> This notebook is part of a series presented at the **[Workshop on Remote Sensing and Critical Mineral Discovery](https://atmos.utah.edu/critical_minerals_imaging_workshop/index.php)** presented at the University of Utah on September 29, 2026.

Authors: K. Dana Chadwick<sup>1</sup>, Philip G. Brodrick<sup>1</sup>, Erik A. Bolch<sup>2</sup>, Rupesh Shrestha<sup>3</sup>, and Noah J. Christensen<sup>4</sup>

1. NASA Jet Propulsion Laboratory, California Institute of Technology.
2. KBR Inc., contractor to the USGS Earth Observation Research and Science Center, NASA Land Processes Distributed Active Archive Center.
3. Oak Ridge National Laboratory Distributed Active Archive Center.
4. Utah Geological Survey.

## Summary

The EMIT visible-to-shortwave-infrared (VSWIR) imaging spectrometer measures reflected sunlight
across many adjacent wavelength bands in the 380 - 2500 nm range. In this notebook we use the
`earthaccess` python library to search for and stream the EMIT L2B Mineral Identification, Band
Depth and Uncertainty product ([EMITL2BMIN.001](https://doi.org/10.5067/EMIT/EMITL2BMIN.001)) and
the EMIT L2A Reflectance product ([EMITL2ARFL.001](https://doi.org/10.5067/EMIT/EMITL2ARFL.001))
over the **White Mica AOI** on the east flank of the Wah Wah Range, Beaver County, Utah. We use
the mineralogy product to locate white micas, return to the reflectance to measure the position of
the Al-OH absorption feature near 2.2 µm at every white-mica pixel, and then test that measurement
against a coincident AVIRIS-5 airborne flight line from the ORNL DAAC, which resolves the inside of
an EMIT pixel at roughly 10 m.

| | |
|---|---|
| Companion notebook (1) | `01_alunite_temporal_aggregation.ipynb`: aggregates EMIT L2B_MIN across many dates and tests the result against UGS surface geochemistry, over the Southern Pine Valley AOI. |
| This notebook (2) | Identifies white micas with EMIT, measures the Al-OH absorption center, the diagnostic that carries the compositional information, and compares that measurement between EMIT (~60 m, spaceborne) and AVIRIS-5 (~10 m, airborne) reflectance over the same ground. |

### Background

#### From mineral labels to mineral chemistry

Notebook 1 used EMIT's L2B_MIN product as a classifier: each pixel gets a label, the labels are
mapped, and the maps are aggregated. For a mineral that either is or is not present, that is close to
sufficient.

White mica is not one mineral. It is a solid-solution series running from muscovite
`KAl₂(AlSi₃O₁₀)(OH)₂` toward phengite, as Al in the octahedral site is progressively replaced by Fe
and Mg. That substitution moves the wavelength of the Al-OH absorption feature near 2.2 µm: more Al
gives a shorter wavelength (~2.185–2.195 µm), more Fe–Mg a longer one (~2.205–2.215 µm). The shift
between those end members is only about 30 nm, but it is systematic, and it is a proxy for the chemistry of the
fluid the mica grew from, chiefly its pH, since Al mobility in hydrothermal fluid is strongly
pH-dependent. Meyer et al. (2022) calibrate the shift: a 1 nm change in the position of the 2200 nm
white-mica feature corresponds to a change of approximately 1.05% in octahedral Al, and shifts of a
few nanometres are geologically significant across base-metal sulfide, epithermal, porphyry and
orogenic gold systems. Graham et al. (2018) measured the same feature in airborne imaging
spectroscopy over unmined porphyry Cu deposits in the eastern Alaska Range, where white-mica Al-OH
position both outlined the known deposits and led to mineralized rock away from them.

The classifier answer, "there is white mica here", therefore discards the compositional information.
Recovering it means going back to the reflectance, removing the continuum, and measuring where the
absorption bottoms out, which is what §5 and §7 do.

#### Why compare two sensors?

Measuring a feature position rather than reading a label makes spatial resolution a question of
whether the measurement is valid. A 60 m EMIT pixel covers about 26 AVIRIS-5 pixels. Whatever
mixture of minerals, lichen, shadow and bare rock lies inside that footprint, EMIT reports one
area-weighted spectrum for it. Reflectance mixing is not linear in the way absorption-feature
parameters are: mixing two micas with centers at 2.190 and 2.210 µm does not reliably give a feature
at 2.200 µm. It can give a broadened feature, an asymmetric one, or one dominated by whichever
component is brighter.

AVIRIS-5 allows a direct test, because it resolves the inside of an EMIT pixel. For each site we
extract every airborne spectrum within the spaceborne footprint, measure each one's absorption
center, and compare: does the EMIT center land at the mean of those, at the median, or somewhere
biased, and how much real variation does the 60 m pixel conceal? That question bounds the spaceborne
measurement, and it is answerable only with coincident higher-resolution data.

#### The study area: white-mica alteration in a Blawn Formation lithocap

The White Mica AOI (`data/WhiteMica_AOI.shp`, ~38.24–38.42° N, 113.54–113.68° W, roughly
12 × 20 km on the east flank of the Wah Wah Range) encloses the white-mica (sericite/illite)
alteration developed in Miocene Blawn Formation rhyolites. This is the near-neutral part of a
lithocap system, grading outward and upward in pH from the acidic core, and it is where the
compositional gradient lives. The lithophile critical-metal suite (Be, Li, Mo, Sn, W, REE, U, F) of
these evolved A-type rhyolites sits in and around this alteration.

Keeping the AOI tight to the white-mica ground means everything from §4 on maps one mineral
group, which keeps the notebook on the measurement rather than on classification bookkeeping, and
the smaller footprint means fewer pixels to read.

### References

- Clark, R.N. & Roush, T.L. (1984) Reflectance spectroscopy: quantitative analysis techniques for
  remote sensing applications. *Journal of Geophysical Research* 89, 6329. (continuum removal)
- Meyer, J.M. et al. (2022) Quantifying mineralogy and chemistry with imaging spectroscopy.
  *Remote Sensing of Environment* 275, 113000. (white-mica spectroscopy review)
- Graham, G.E. et al. (2018) Application of imaging spectroscopy for mineral exploration in Alaska.
  *Economic Geology* 113, 489. (Al-OH position in porphyry exploration)
- Lindsey, D.A. & Osmonson, L.M. (1978) USGS OFR 78-114. (Blawn Mountain / Wah Wah alunite)
- Barkoff, D. (2022) PhD dissertation, University of Nevada Las Vegas. doi:10.34917/35777457
- Portela, B. et al. (2025) *Ore Geology Reviews* 182, 106673.

Copies of the openly available items are in `references/`.

## Contents

- [1 Setup](#section-1)
    - [1.1 Import libraries](#section-1-1)
    - [1.2 Authenticate with Earthdata Login](#section-1-2)
- [2 Geological context and study area](#section-2)
    - [2.1 Define the white-mica study area](#section-2-1)
    - [2.2 Visualize the study area](#section-2-2)
- [3 Search and access EMIT observations](#section-3)
    - [3.1 Search for mineralogy granules](#section-3-1)
    - [3.2 Screen by cloud cover and time of year](#section-3-2)
    - [3.3 Rank candidate scenes by white-mica richness](#section-3-3)
    - [3.4 Resolve the companion asset URLs](#section-3-4)
- [4 Identify white micas with EMIT](#section-4)
    - [4.1 Stream and orthorectify the mineralogy cube](#section-4-1)
    - [4.2 Mineral-ID to name lookup](#section-4-2)
    - [4.3 What "white mica" means in this library](#section-4-3)
    - [4.4 Quality gates](#section-4-4)
    - [4.5 Map the white-mica detections](#section-4-5)
- [5 Measure the Al-OH absorption center from EMIT reflectance](#section-5)
    - [5.1 Load L2A reflectance with a per-pixel cloud mask](#section-5-1)
    - [5.2 Continuum removal, step by step](#section-5-2)
    - [5.3 Absorption-center measurement at every white-mica pixel](#section-5-3)
- [6 Search and access AVIRIS-5 airborne reflectance](#section-6)
    - [6.1 Search for flight lines over the study area](#section-6-1)
    - [6.2 Select one flight line](#section-6-2)
    - [6.3 Cache the flight line, with an integrity check](#section-6-3)
    - [6.4 How pixels are read from the cube, and why the access pattern matters](#section-6-4)
    - [6.5 One up-front read for display and validity](#section-6-5)
- [7 Compare EMIT and AVIRIS-5 across scales](#section-7)
    - [7.1 Choose comparison sites that span the composition range](#section-7-1)
    - [7.2 The number of AVIRIS pixels within one EMIT pixel](#section-7-2)
    - [7.3 Where the sites sit, in true color at both scales](#section-7-3)
    - [7.4 Raw reflectance at both scales](#section-7-4)
    - [7.5 Continuum-removed, on a shared baseline](#section-7-5)
    - [7.6 The same measurement, mapped at both scales](#section-7-6)
    - [7.7 The full-scene picture: detection next to composition](#section-7-7)
- [8 Interpretation and next steps](#section-8)

## Learning objectives

By the end of this notebook you will be able to:

1. Identify white micas with EMIT L2B_MIN, and say precisely what the "white mica" class
   collapses together (muscovite, illite, sericite, and their Al-rich vs Fe–Mg variants), and what
   that collapse costs.
2. Select an appropriate EMIT scene by cloud cover, season-matching to an airborne campaign,
   and a mineral-richness check to ensure there isn't excessive interference from vegetation, snow, or other cover types.
3. Remove the spectral continuum (Clark & Roush, 1984) and explain what it does and does not
   normalise away.
4. Measure Al-OH feature position, depth and FWHM and interpret position as a white-mica
   composition and fluid-pH proxy across the muscovite–phengite range (~2.185–2.215 µm).
5. Map absorption-center position across a scene to turn a categorical mineral map into a
   continuous fluid-chemistry proxy.
6. Access AVIRIS-5 L2A reflectance from the ORNL DAAC, including reading the GSD from the file
   rather than assuming it.
7. Quantify the effect of spatial scale on a spectral measurement: extract every AVIRIS pixel
   inside one EMIT pixel, compare their absorption-center distribution against the single EMIT
   value, and state when the 60 m measurement can and cannot be read as mineral chemistry.
8. Recognise the failure modes: sub-pixel mixing, band-depth dilution, geolocation offsets
   between platforms, and non-coincident acquisition dates.

## Prerequisites

1. A free NASA Earthdata Login: <https://urs.earthdata.nasa.gov/>. `earthaccess` will create a
   `~/.netrc` for you the first time you log in. The same login covers both the LP DAAC (EMIT) and
   the ORNL DAAC (AVIRIS-5).
2. The `seg_critical_minerals` conda environment (see `environment.yml` / `README.md`):
   ```bash
   conda env create -f environment.yml
   conda activate seg_critical_minerals
   ```

Data volume. Unlike notebook 1, this notebook works with full reflectance cubes. The
two platforms are handled differently. The EMIT granules are **streamed**: `earthaccess.open()`
serves them over HTTPS range requests, only the AOI subset is brought into memory, and nothing is
written to `./data/`. The AVIRIS-5 flight line is **downloaded and cached**, because the windowed
reads in section 6.4 are what keep the cross-scale comparison responsive.

| | transferred | written to `./data/` |
|---|---|---|
| EMIT L2B_MIN, per candidate scene probed in section 3.3 | ~40 MB subset | nothing (streamed) |
| EMIT L2B_MIN + L2B_MINUNCERT, selected scene | ~80 MB subset | nothing (streamed) |
| EMIT L2A_RFL + L2A_MASK | ~2 GB | nothing (streamed) |
| AVIRIS-5 L2A reflectance, one flight line | ~2.2 GB | ~2.2 GB |

Allow roughly 3 GB of disk and 15 to 30 minutes for a first run, most of it transfer time. Because
the EMIT reads are not cached, re-running section 5 re-transfers the reflectance granule, whereas
the AVIRIS flight line, its VRT descriptor and the ROI `.npz` caches persist, so second and later
runs of section 6 onward are substantially faster. The AVIRIS file is addressed through a GDAL VRT
and read in windows, so no run holds a whole cube in memory.

<a id="section-1"></a>

## 1. Setup

<a id="section-1-1"></a>

### 1.1 Import libraries

Maps in this notebook are drawn with **hvPlot**, which renders through HoloViews, GeoViews and
Bokeh and gives pan, zoom and hover on a tiled basemap. Line plots, histograms and the projected
AVIRIS-5 panels stay in **matplotlib**, where a fixed, publication-style figure is the point. The
two local modules, `emit_tools` and `spectral_utils`, are adapted from the NASA EMIT and AVIRIS
data-resource repositories and live in `modules/`.

In [ ]:
import datetime as _dt
import re as _re
import sys
import time
import warnings
from contextlib import closing
from pathlib import Path
from urllib.parse import urlparse

import earthaccess
import geopandas as gpd
import h5py                   # cheap integrity check on the cached AVIRIS-5 download
import holoviews as hv
import hvplot.pandas          # registers the .hvplot accessor on GeoDataFrame
import hvplot.xarray          # registers the .hvplot accessor on DataArray / Dataset
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio               # windowed AVIRIS-5 pixel reads (see section 6.4)
import rioxarray as rxr       # AVIRIS-5 VRT reader, used for the cube's coordinate/CRS metadata
import xarray as xr
from osgeo import gdal        # gdal.BuildVRT writes the AVIRIS-5 VRT descriptor
from pyproj import Proj, Transformer
from rasterio.windows import Window
from shapely.geometry import Point, shape
from shapely.geometry.polygon import orient

gdal.UseExceptions()                            # raise Python exceptions rather than return None
gdal.PushErrorHandler("CPLQuietErrorHandler")   # suppress GDAL's verbose non-fatal messages

# Local helper modules adapted from the NASA EMIT and AVIRIS data-resource repositories.
sys.path.append("modules")
import emit_tools as et        # emit_xarray, ortho_xr, spatial_subset, quality_mask
import spectral_utils as su    # continuum removal, Al-OH feature metrics

# Library deprecation notices are silenced to keep the tutorial output readable; remove this line
# when debugging an unexpected result.
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 60)     # keep wide DataFrame columns legible
hv.extension("bokeh")                         # HoloViews/hvPlot render through Bokeh

DATA_DIR = Path("data")
FIG_DIR = Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

print("xarray", xr.__version__, "| earthaccess", earthaccess.__version__,
      "| hvplot", hvplot.__version__, "| GDAL", gdal.VersionInfo("RELEASE_NAME"))

<a id="section-1-2"></a>

### 1.2 Authenticate with Earthdata Login

A free NASA [Earthdata Login](https://urs.earthdata.nasa.gov/) account is required. `persist=True`
writes a `~/.netrc` entry so credentials are entered once and later runs reuse them silently. The
same credentials work for both the LP DAAC (EMIT) and the ORNL DAAC (AVIRIS-5).

In [ ]:
# --- Authenticate to NASA Earthdata ------------------------------------------------------------
# Every EMIT and AVIRIS-5 read below is gated behind a (free) NASA Earthdata Login. earthaccess
# handles the OAuth handshake for us: persist=True writes a ~/.netrc token so you only log in once
# and later reruns reuse it silently. On first run this prompts for
# your Earthdata username/password (or reads existing ~/.netrc / EARTHDATA_* environment variables).
auth = earthaccess.login(persist=True)
print("Earthdata authenticated:", auth.authenticated)

<a id="section-2"></a>

## 2. Geological context and study area

The study area is read from a shapefile and drives everything downstream: the CMR searches, the
spatial subsets, the AVIRIS-5 clip, and every map extent.

<a id="section-2-1"></a>

### 2.1 Define the white-mica study area

The ROI polygon, `data/WhiteMica_AOI.shp`, a single rectangle drawn around the white-mica
alteration, drives every data search, every plot extent, and the AVIRIS clip below. It is an
ESRI shapefile rather than GeoJSON, so read it the same way (`geopandas` handles both) but do check
its CRS: a shapefile carries its projection in a sidecar `.prj`, and a silently-projected ROI would
send meaningless coordinates to CMR.

CMR also expects a counter-clockwise exterior ring, so we normalise the winding order with
`shapely.orient` before handing the coordinates to `earthaccess`: a clockwise ring is silently
interpreted as the polygon's complement, i.e. everywhere on Earth except your study area, which
returns either nothing or nonsense.

In [ ]:
# --- Study-area ROI: the White Mica AOI ---------------------------------------------------------
ROI_PATH = DATA_DIR / "WhiteMica_AOI.shp"
roi_gdf = gpd.read_file(ROI_PATH)
if roi_gdf.crs is None:
    raise SystemExit(f"{ROI_PATH.name} has no CRS (.prj missing?) - cannot trust its coordinates")
if roi_gdf.crs.to_epsg() != 4326:
    roi_gdf = roi_gdf.to_crs(epsg=4326)      # CMR, contextily and every plot below want lon/lat

# CMR wants a COUNTER-CLOCKWISE exterior ring; orient(sign=1.0) guarantees it regardless of how the
# polygon was digitised. Note the (lon, lat) ordering that earthaccess expects.
roi_poly = orient(roi_gdf.geometry.iloc[0], sign=1.0)
roi_coords = list(roi_poly.exterior.coords)
bbox = tuple(roi_gdf.total_bounds)           # (W, S, E, N), reused by the searches that follow

# Approximate the AOI dimensions in kilometres: one degree of latitude spans ~111.13 km, and one
# degree of longitude ~111.32 km scaled by cos(latitude), evaluated at the AOI mid-latitude.
_w, _s, _e, _n = bbox
_km_x = (_e - _w) * 111.32 * np.cos(np.deg2rad(0.5 * (_s + _n)))
_km_y = (_n - _s) * 111.13
print(f"ROI       : {ROI_PATH}  ({len(roi_gdf)} feature, CRS {roi_gdf.crs.to_string()})")
print(f"extent    : lon {_w:.5f} .. {_e:.5f},  lat {_s:.5f} .. {_n:.5f}")
print(f"footprint : ~{_km_x:.0f} x {_km_y:.0f} km   |   exterior ring CCW: {roi_poly.exterior.is_ccw}")

<a id="section-2-2"></a>

### 2.2 Visualize the study area

The AOI outline is drawn over a labelled topographic basemap (Esri World Topo) so the Wah Wah Range
crest, the drainages and the local place names are visible before any spectral data is loaded.
hvPlot reprojects the polygon to the Web Mercator frame the tiles are served in, so the
latitude/longitude geometry is passed through unchanged.

**Figure 1. Study-area locator:** the White Mica AOI over a labelled topographic basemap. Pan and
zoom to place the AOI relative to the range crest and the Blawn Mountain area.

In [ ]:
# `geo=True` tells hvPlot the data are geographic, so it reprojects to the Web Mercator frame that
# the tile service uses. The same outline is reused by every later map, so it is kept in a variable.
OUTLINE_OPTS = dict(geo=True, fill_alpha=0, line_color="red", line_width=2.5, hover=False)
roi_outline = roi_gdf.hvplot.polygons(**OUTLINE_OPTS)

study_area_figure = roi_gdf.hvplot.polygons(tiles="EsriWorldTopo", **OUTLINE_OPTS).opts(
    title="White Mica AOI: Wah Wah Range, Beaver County, Utah",
    width=760, height=660, padding=0.15, xaxis=None, yaxis=None,
)
study_area_figure

<a id="section-3"></a>

## 3. Search and access EMIT observations

| Product | short_name / concept-id |
|---------|-------------------------|
| EMIT L2B Mineralogy | `EMITL2BMIN` · `C2408034484-LPCLOUD` · DOI 10.5067/EMIT/EMITL2BMIN.001 |
| EMIT L2B Mineralogy Uncertainty | `EMITL2BMINUNCERT` (bundled in the same granule) |
| EMIT L2A Reflectance & Mask | `EMITL2ARFL` · DOI 10.5067/EMIT/EMITL2ARFL.001 |
| AVIRIS-5 L2A Reflectance | `C4184090548-ORNL_CLOUD` (section 6) |

L2B_MIN is produced by the USGS Tetracorder expert system, which matches each pixel's
continuum-removed reflectance against a spectral library. Each pixel receives up to two
identifications: group 1 for the ~1 µm (electronic / Fe) region and group 2 for the
2.0–2.5 µm region where the Al-OH phyllosilicates and sulfates live, each with a band depth, plus
a fit value in the companion MINUNCERT file.

For this notebook L2B_MIN plays a supporting role: it tells us where the white micas are, so
that the reflectance analysis in sections 5 and 7 is aimed at pixels that actually contain the
mineral we want to characterise. The measurement itself comes from L2A reflectance.

<a id="section-3-1"></a>

### 3.1 Search for mineralogy granules

`earthaccess.search_data` queries NASA's Common Metadata Repository (CMR). Passing the AOI as a
polygon returns every L2B_MIN granule whose footprint intersects it, across the whole mission
record, which is the candidate pool the next two subsections narrow to one scene.

In [ ]:
# --- Search NASA CMR for EMIT L2B mineralogy granules over the ROI -----------------------------
# EMIT L2BMIN ("Estimated Mineral Identification and Band Depth") is the spaceborne mineral-mapping
# product we build section 4 on. We query by the collection's CMR concept-id (a stable handle for
# "this exact product + version") and the ROI polygon, so CMR returns only granules that overlap our
# study area. count=500 is just a safe upper bound on how many results to page back.
EMIT_L2BMIN_CID = "C2408034484-LPCLOUD"

emit_results = earthaccess.search_data(
    concept_id=EMIT_L2BMIN_CID,     # EMIT L2B mineralogy, LP DAAC cloud collection
    polygon=roi_coords,             # CCW ROI vertices from the cell above (CMR wants CCW)
    count=500,                      # max granules to return (we expect far fewer)
)
# Each result is a granule spanning some date/time; the next cells screen these by cloud + season +
# mineral richness to pick the single scene the tutorial analyses.
print(f"EMIT L2BMIN granules over ROI: {len(emit_results)}")

<a id="section-3-2"></a>

### 3.2 Screen by cloud cover and time of year

EMIT granules carry a per-granule cloud-cover value that is well suited to scene screening. CMR
reports it for every EMIT granule as the standard UMM `CloudCover` field (a scene-average percent,
0-100), which `earthaccess` exposes at `result["umm"]["CloudCover"]`. Because clouds and their
shadows corrupt the retrieved surface reflectance, and therefore the L2B_MIN mineral identifications
built on it, this field is used to drop any scene with more than 25% cloud cover before choosing
one to work with.

> Two complementary cloud controls are applied in this notebook. The scene-level `CloudCover`
> filter used here is a coarse first pass that rejects mostly-cloudy granules. It does not tell us
> which pixels are cloudy, so in §5.1 we additionally apply the per-pixel L2A cloud mask to the
> reflectance. The coarse scene-level screen is therefore applied first and the per-pixel mask
> second.

The surviving scenes are then ranked by how close they fall to the AVIRIS-5 time of year. This
notebook compares EMIT against an AVIRIS-5 flight line, so ideally the two platforms see the surface
in as similar a state as possible: the same snow-free, dry-vegetation and illumination conditions.
The AVIRIS-5 flight date is read from a quick CMR probe of the AVIRIS-5 collection, whose granule
ids encode the flight date.

In [ ]:
# --- Granule metadata accessors -----------------------------------------------------------------
def granule_name(result):
    """Producer granule id, e.g. EMIT_L2B_MIN_001_20230730T... ."""
    try:
        return result["umm"]["GranuleUR"]
    except Exception:
        return result["meta"]["native-id"]


def cloud_cover(result):
    """Per-granule cloud-cover percent (0-100) from the CMR UMM `CloudCover` field."""
    try:
        return float(result["umm"]["CloudCover"])
    except (KeyError, TypeError, ValueError):
        return np.nan   # treat "unknown" as not-screened-out below


def scene_datetime(result):
    """Acquisition datetime of an EMIT granule, parsed from its granule id timestamp
    (..._YYYYMMDDTHHMMSS_...). Falls back to the CMR temporal start if the id has no timestamp."""
    m = _re.search(r"_(\d{8}T\d{6})_", granule_name(result))
    if m:
        return _dt.datetime.strptime(m.group(1), "%Y%m%dT%H%M%S")
    try:
        s = result["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
        return _dt.datetime.fromisoformat(s.replace("Z", "+00:00")).replace(tzinfo=None)
    except Exception:
        return None


def asset_url(result, marker):
    """Find the one netCDF asset in a granule by its filename marker, e.g. '_L2B_MIN_'.

    Asset URLs always come from the granule metadata; hand-building them is how a notebook
    silently breaks on the next product version."""
    try:
        urls = result.data_links()
    except Exception:
        urls = [u["URL"] for u in result["umm"]["RelatedUrls"] if u.get("Type") == "GET DATA"]
    hits = [u for u in urls if marker in Path(urlparse(u).path).name and u.endswith(".nc")]
    if len(hits) != 1:
        raise ValueError(f"expected one {marker} asset, found {len(hits)}")
    return hits[0]


# --- Temporal target: match the EMIT scene to the AVIRIS-5 flight's time of year -----------------
# A quick CMR query on the AVIRIS-5 L2A collection over the AOI gives the reference date, taken from
# the granule id, which encodes the flight date. If the search returns nothing, fall back to the
# known campaign window.
AVIRIS5_L2A_RFL_CID = "C4184090548-ORNL_CLOUD"
try:
    _av_probe = earthaccess.search_data(concept_id=AVIRIS5_L2A_RFL_CID, polygon=roi_coords, count=50)
    _av_dates = []
    for _r in _av_probe:
        _m = _re.search(r"(\d{8})[tT]\d{6}", (_r["umm"]["GranuleUR"] if "umm" in _r else str(_r)))
        if _m:
            _av_dates.append(_dt.datetime.strptime(_m.group(1), "%Y%m%d"))
    if _av_dates:
        AVIRIS_TARGET_DATE = sorted(_av_dates)[len(_av_dates) // 2]   # median flight date
    else:
        raise ValueError("no AVIRIS dates parsed")
except Exception as _e:
    AVIRIS_TARGET_DATE = _dt.datetime(2025, 7, 23)   # known campaign window fallback
    print("AVIRIS date probe unavailable; using fallback target", AVIRIS_TARGET_DATE.date(), f"({_e})")
AVIRIS_TARGET_DOY = AVIRIS_TARGET_DATE.timetuple().tm_yday
print(f"AVIRIS-5 reference date for scene matching: {AVIRIS_TARGET_DATE.date()} "
      f"(day-of-year {AVIRIS_TARGET_DOY})")

# --- Scene-level cloud screen -------------------------------------------------------------------
cc = np.array([cloud_cover(r) for r in emit_results])
print("cloud cover reported for {}/{} granules; range {:.0f}-{:.0f}%".format(
    int(np.isfinite(cc).sum()), len(cc),
    np.nanmin(cc) if np.isfinite(cc).any() else float("nan"),
    np.nanmax(cc) if np.isfinite(cc).any() else float("nan")))

# Keep only scenes with <=25% cloud cover (unknown values are kept, not silently dropped).
CLOUD_MAX = 25.0
keep = ~(cc > CLOUD_MAX)
emit_clear = [r for r, k in zip(emit_results, keep) if k]
print(f"granules with <= {CLOUD_MAX:.0f}% cloud cover: {len(emit_clear)} of {len(emit_results)}")


# --- Seasonal ranking ---------------------------------------------------------------------------
def _doy_gap(dt):
    """Smallest circular day-of-year distance (0-182.5) between a date and the AVIRIS target."""
    if dt is None:
        return 999
    g = abs(dt.timetuple().tm_yday - AVIRIS_TARGET_DOY)
    return min(g, 365 - g)


def season_score(result):
    """Sort key (season gap in days, |year gap|, cloud %); lower is closer to the AVIRIS flight."""
    dt = scene_datetime(result)
    season = _doy_gap(dt)
    yeargap = abs(dt.year - AVIRIS_TARGET_DATE.year) if dt is not None else 999
    return (season, yeargap, np.nan_to_num(cloud_cover(result), nan=999))

<a id="section-3-3"></a>

### 3.3 Rank candidate scenes by white-mica richness

Season is not enough on its own. L2B_MIN retrievals vary scene to scene with illumination,
vegetation, dust and surface moisture, and a clear, well-timed scene is of little use here if EMIT
did not identify white mica over the AOI that day. A scene carrying only a handful of white-mica
pixels leaves the §4.5 map and the §7 multi-scale comparison with too little to work with. Richness is
therefore a hard requirement, and season sets the order in which candidates are tried:

1. cloud-screen (done in §3.2, drop > 25%);
2. rank the survivors by season, closest to the AVIRIS-5 day-of-year first, preferring the same year;
3. walk that ranked list, reading each candidate's L2B_MIN and counting how many AOI pixels carry
   a white-mica identification in either Tetracorder group (raw ID membership, no quality cut);
4. select the first, that is the most season-matched, scene that clears the richness threshold.

The cell prints the probed shortlist with each scene's white-mica count, season gap, year gap and
cloud percentage, so the whole decision is visible in the output.

> **Granules are streamed, not downloaded.** `earthaccess.open()` returns a file-like object backed
> by range requests against the archive, and `emit_tools` accepts it in place of a local path. The
> AOI is a small fraction of a 75 km EMIT swath, so subsetting before orthorectification keeps each
> probe to a few tens of megabytes. Nothing is cached, so the selected scene is read once here and
> again in §4.1; that is the cost of not filling the working directory with netCDF files.

In [ ]:
def read_roi_subset(url, qmask=None):
    """Stream one EMIT granule and return the AOI subset on a lat/lon grid.

    The three emit_tools calls run in the order that is cheapest:
      1. emit_xarray(ortho=False)  open the granule without orthorectifying yet (fast, low memory);
      2. spatial_subset(roi_gdf)   crop to the AOI while still in raw sensor space, so step 3 warps
                                   only the pixels this notebook uses, not the whole scene;
      3. ortho_xr()                apply EMIT's geometric lookup table (GLT) to warp to lat/lon.
    `closing` releases the streamed handle as soon as the subset is in memory. DAAC reads fail
    transiently often enough that a bare retry with backoff is worth the four lines."""
    for attempt in range(3):
        try:
            with closing(earthaccess.open([url], provider="LPCLOUD")[0]) as file:
                raw = et.emit_xarray(file, ortho=False, qmask=qmask)
                try:
                    return et.ortho_xr(et.spatial_subset(raw, roi_gdf)).load()
                finally:
                    raw.close()
        except Exception as error:
            if attempt == 2:
                raise
            print(f"    retrying after error: {error}")
            time.sleep(2 ** attempt)


# Library indices (into the L2B_MIN mineral_id integers) for the white-mica entries, read from the
# EMIT mineral grouping matrix. Section 4.3 rebuilds the same list, with discussion, for the
# quality-gated analysis; here it only needs to answer "is this pixel a white mica?".
_gate_matrix = pd.read_csv(DATA_DIR / "mineral_grouping_matrix_20230503.csv")


def _gate_is_white_mica(name):
    n = str(name).lower()
    # Ammonium- and chlorite-bearing entries are excluded here for the same reasons given in
    # section 4.3. The two definitions must agree, or this gate would rank scenes on a different
    # population than the one the notebook goes on to measure.
    return (any(k in n for k in ("muscovite", "illite", "sericite"))
            and not any(x in n for x in ("ammonio", "ammonium", "chlorite")))


_gate_ids = [int(i) for i, nm in zip(_gate_matrix["Index"], _gate_matrix["Name"])
             if _gate_is_white_mica(nm)]

RICHNESS_MIN = 300   # min AOI pixels with a white-mica ID (~1 km2 at 60 m) to call a scene usable
MAX_PROBE = 8        # cap how many candidates are streamed and probed (season-matched first)


def white_mica_pixel_count(result):
    """Stream this scene's L2B_MIN, clip to the AOI, and count pixels whose group-1 OR group-2
    mineral id is a white-mica library entry (raw membership, no quality threshold). Returns the
    count, or -1 if the scene could not be read."""
    try:
        ds = read_roi_subset(asset_url(result, "_L2B_MIN_"))
        want = (np.isin(ds["group_1_mineral_id"].values, _gate_ids)
                | np.isin(ds["group_2_mineral_id"].values, _gate_ids))
        ds.close()
        return int(np.nansum(want))
    except Exception as e:
        print(f"    could not probe {granule_name(result)}: {e}")
        return -1


# Walk season-ranked candidates, probing white-mica richness until one clears the bar.
ranked = sorted(emit_clear, key=season_score)
print(f"\nProbing scenes (season-matched first) for white-mica richness, need >= {RICHNESS_MIN} "
      f"AOI pixels with a white-mica ID:")
sel_result = None
_shortlist = []
for r in ranked[:MAX_PROBE]:
    dt = scene_datetime(r)
    s, y, c = season_score(r)
    n_wm_raw = white_mica_pixel_count(r)
    _shortlist.append((r, dt, s, y, c, n_wm_raw))
    flag = "OK" if n_wm_raw >= RICHNESS_MIN else ("sparse" if n_wm_raw >= 0 else "unreadable")
    print(f"  {granule_name(r):52s} {str(dt.date()) if dt else '????':>10}  "
          f"season {s:>3.0f} d, yr {y}, {c:4.0f}% cloud  ->  {n_wm_raw:>6} white-mica px  [{flag}]")
    if n_wm_raw >= RICHNESS_MIN:
        sel_result = r
        break

# Fallback: if none of the probed candidates cleared the bar, take the richest one seen.
if sel_result is None:
    readable = [t for t in _shortlist if t[5] >= 0]
    if not readable:
        raise RuntimeError("No EMIT scene over the AOI could be read for the richness gate.")
    best = max(readable, key=lambda t: t[5])
    sel_result = best[0]
    print(f"\nNo scene reached {RICHNESS_MIN} white-mica px within the first {MAX_PROBE} candidates; "
          f"falling back to the richest one probed ({best[5]} px).")

sel_name = granule_name(sel_result)
_sel_dt = scene_datetime(sel_result)
_sel_season, _sel_year, _sel_cloud = season_score(sel_result)
_day_gap = abs((_sel_dt - AVIRIS_TARGET_DATE).days) if _sel_dt else None
print(f"\nSelected: {sel_name}")
print(f"  acquired {_sel_dt.date() if _sel_dt else '?'}  |  {cloud_cover(sel_result):.0f}% cloud  |  "
      f"{_day_gap} d from the AVIRIS-5 campaign reference date  |  season gap {_sel_season:.0f} d, "
      f"year gap {_sel_year}")
print("  (the most season-matched clear scene that actually contains white mica over the AOI.)")

<a id="section-3-4"></a>

### 3.4 Resolve the companion asset URLs

Four assets are needed. The L2B_MIN granule bundles both the mineral identifications and their
uncertainty companion, which carries the Tetracorder fit values gated in §4.4. The L2A reflectance
and its per-pixel mask live in a separate granule, matched to this scene by the orbit and scene
numbers at the end of the granule id.

In [ ]:
min_url = asset_url(sel_result, "_L2B_MIN_")
minunc_url = asset_url(sel_result, "_L2B_MINUNCERT_")

# The L2A reflectance is a separate granule (EMITL2ARFL). Find the one whose granule id shares this
# scene's orbit/scene suffix (e.g. '..._2223014_011'), then take its RFL and MASK asset URLs.
scene_suffix = "_".join(sel_name.split("_")[-2:])   # e.g. '2223014_011'
rfl_results = earthaccess.search_data(short_name="EMITL2ARFL", polygon=roi_coords, count=500)
rfl_match = [r for r in rfl_results if granule_name(r).endswith(scene_suffix)][0]
rfl_url = asset_url(rfl_match, "_L2A_RFL_")
mask_url = asset_url(rfl_match, "_L2A_MASK_")

for label, u in [("MIN", min_url), ("MINUNCERT", minunc_url), ("RFL", rfl_url), ("MASK", mask_url)]:
    print(f"{label:9s}", u.split("/")[-1])

<a id="section-4"></a>

## 4. Identify white micas with EMIT

Four steps: orthorectify the mineral cube onto a lat/lon grid, translate the integer mineral IDs into
names, decide which library entries count as "white mica", and gate the detections on quality before
mapping them. Only the white-mica entries of the spectral library are used; the remaining entries are
carried through the read but never mapped.

<a id="section-4-1"></a>

### 4.1 Stream and orthorectify the mineralogy cube

EMIT data ship in raw sensor space: a pushbroom array that is not yet map-projected. The
`read_roi_subset` helper defined in §3.3 opens the granule, crops it to the AOI while still in
sensor space, and only then applies the geometric lookup table, which is far cheaper than
orthorectifying the whole 75 km swath.

In [ ]:
# --- Load the L2BMIN mineralogy cube, clip to the ROI, and orthorectify to lat/lon -------------
# Streamed, subset and orthorectified by the helper defined above.
ds_min = read_roi_subset(min_url)
#print(ds_min)                                      # uncomment to inspect dims/vars: group_1/2 mineral_id, band_depth, ...

<a id="section-4-2"></a>

### 4.2 Mineral-ID to name lookup

`emit_xarray` exposes the L2B_MIN library metadata on a `mineral_name` dimension. We turn those
coordinate variables into a small lookup table. ID 0 means "no identification" and is not stored as
a library row, so we prepend it explicitly; otherwise every ID is off by one, which is the kind of
bug that produces a plausible-looking but entirely wrong mineral map.

In [ ]:
# --- Recover the mineral-name lookup table that the integer mineral ids point into --------------
# L2BMIN does not store mineral names per pixel; it stores integer ids (group_1_mineral_id, etc.)
# that index into a spectral library carried as coordinate variables on the file. Here we pull every
# such library coordinate (those defined along the `mineral_name` dimension) into a tidy DataFrame so
# we can map id -> human-readable name later.
min_df = pd.DataFrame(
    {v: ds_min[v].values for v in ds_min.coords if "mineral_name" in ds_min[v].dims}
)
# Id 0 means "No_Match" (EMIT identified no library mineral for that pixel) and is not stored as a
# library row, so we prepend it explicitly at index 0 so positional lookups line up with the ids.
zero_row = {c: (0 if c in ("index",) else ("No_Match" if c == "mineral_name" else np.nan))
            for c in min_df.columns}
min_df.loc[-1] = zero_row                       # add the No_Match row (temporary index -1)
min_df = min_df.sort_index().reset_index(drop=True)   # move it to row 0, renumber cleanly
print(min_df.head())
print("\nlibrary entries:", len(min_df))

<a id="section-4-3"></a>

### 4.3 What "white mica" means in this library

This notebook maps one mineral group, so the classification step is correspondingly small: pull
the library rows whose names denote a white mica (muscovite, illite, sericite and their mixtures) and
ignore the rest of the library entirely.

The cell below lists every entry that qualifies: muscovite, illite, and several compositional
variants of muscovite that Tetracorder distinguishes: low-Al, medium-Al and high-Al muscovite, an
Fe-rich muscovite, two illites, plus two muscovite–pyrophyllite mixtures.

Three entries whose names would otherwise qualify are left out, because section 5.3 measures where
one particular absorption sits and each of them puts something other than a clean Al–OH band into
that measurement.

`Ammonio-Illite/Smectit GDS87` carries "illite" in its name but is an ammonium-bearing phase. Its
deepest absorption between 2.1 and 2.26 µm is the N–H combination band of the NH₄ ion near 2.12 µm
rather than the Al–OH band near 2.20 µm, so its pixels would report the position of a different
vibration. Tetracorder's own file path, `group.2um/smectite_ammonillsmec`, places it with the
smectites rather than in the mica group.

`Muscovite+Chlorite CU91-253D` and `Chlorite+Muscovite CU93-65A` are chlorite mixtures, where the
problem is the continuum rather than the mineral. Chlorite absorbs strongly near 2.25 µm, on the
upper shoulder of the 2.10–2.26 µm window. That drags the long-wavelength anchor of the straight-line
continuum down, and dividing by a continuum tilted that steeply depresses the short-wavelength end of
the result, so the deepest point migrates to the bottom edge of the window and the reported position
is an artefact of the fit. Widening the window to enclose the chlorite band would trade that artefact
for another, since the Al–OH band would then compete with a deeper neighbour.

The grouping step discards the compositional information provided by Tetracorder outputs, but instead
allows the analysis to focus on the absorption center. The compositional information will be
determined by the reflectance instead as part of a continuum. This approach is presented in section 5
where we measure the absorption center directly, and treat it as a continuous quantity rather than
trusting a discrete variant label.

In [ ]:
# --- Which library entries count as "white mica"? -----------------------------------------------
# The grouping matrix ships with the L2B_MIN product documentation and carries one row per library
# spectrum, with `Index` = the integer mineral id used in the product itself.
matrix = pd.read_csv(DATA_DIR / "mineral_grouping_matrix_20230503.csv")

WM_TERMS = ("muscovite", "illite", "sericite")     # the white-mica solid-solution series + its ids

# Two families of entry are excluded even though a white-mica name appears in them, because
# §5.3 measures the position of one specific absorption and neither family delivers a clean Al-OH band:
#
#   "ammonio" - `Ammonio-Illite/Smectit GDS87` is ammonium-bearing, and its deepest feature between
#       2.1 and 2.26 µm is the N-H combination band of the NH4 ion near 2.12 µm rather than Al-OH near
#       2.20 µm. Its Tetracorder path agrees: `group.2um/smectite_ammonillsmec` files it under
#       smectite, alongside the other ammonium phases (buddingtonite, ammonio-jarosite).
#   "chlorite" - the muscovite-chlorite mixtures carry a strong chlorite absorption near 2.25 µm, on
#       the upper shoulder of the 2.10-2.26 µm window. That tilts the straight-line continuum steeply,
#       and dividing by it pushes the deepest point of the result to the bottom edge of the window, so
#       the measured position reflects the continuum fit rather than the mineral.
#
# Because is_white_mica requires a WM_TERMS match first, "chlorite" here removes only the two mica
# mixtures (ids 17 and 177) and leaves chlorite-only entries untouched, which were never candidates.
WM_EXCLUDE_TERMS = ("ammonio", "ammonium", "chlorite")

def is_white_mica(name):
    """True for any library entry whose name denotes a white mica, mixtures included.

    Ammonium- and chlorite-bearing entries are rejected; see the note above for the reasoning."""
    n = str(name).lower()
    return any(t in n for t in WM_TERMS) and not any(x in n for x in WM_EXCLUDE_TERMS)

wm_entries = matrix[matrix["Name"].map(is_white_mica)]
wm_ids = sorted(int(i) for i in wm_entries["Index"])

print(f'{len(wm_entries)} of {len(matrix)} library entries are collapsed into "White mica":\n')
for _, r in wm_entries.iterrows():
    print(f"  id {int(r['Index']):3d}  (group {int(r['Group'])})  {r['Name']}")

# Name the rejected entries explicitly, so the exclusion is visible rather than silent.
_excl = matrix[matrix["Name"].map(lambda s: any(t in str(s).lower() for t in WM_TERMS)
                                  and any(x in str(s).lower() for x in WM_EXCLUDE_TERMS))]
for _, r in _excl.iterrows():
    print(f"  excluded: id {int(r['Index']):3d}  {r['Name']}  ({r['Filename']})")
print("\nMany of these entries encode composition (low-, medium- and high-Al muscovite, Fe-rich) or are")
print("mixtures, so Tetracorder is already attempting the measurement this notebook makes from")
print("the reflectance. The variants are collapsed here because a per-pixel, single-date variant")
print("label is fragile; section 5.3 recovers the composition as a continuous number.")

<a id="section-4-4"></a>

### 4.4 Quality gates

A mineral ID alone is not enough, so we also require the detection to be strong. Two gates:

- band depth ≥ `BAND_DEPTH_MIN`: the absorption feature is deep enough to be real, not noise;
- fit ≥ `FIT_MIN`: the library spectrum actually matched the pixel.

> A scaling detail. The L2B `group_*_fit` is stored scaled by 1/2 and must be multiplied
> by 2 before thresholding. Forget this and a `fit >= 0.4` gate silently becomes `fit >= 0.8`,
> discarding most true detections.

In [ ]:
# --- Build the quality-thresholded white-mica mask ----------------------------------------------
# Beyond "the id is a white mica" we require the detection to be strong. Two gates, from the product
# and its uncertainty companion:
#   - band depth >= BAND_DEPTH_MIN : the absorption feature is actually deep (a real, not faint, id)
#   - fit        >= FIT_MIN        : the library spectrum fit the pixel well (goodness-of-match)
# Stream MINUNCERT through the same subset-then-orthorectify helper.
ds_minunc = read_roi_subset(minunc_url)

BAND_DEPTH_MIN = 0.02   # minimum group band depth to trust a detection
FIT_MIN = 0.40          # minimum library-fit quality to trust a detection


def white_mica_mask(ds, ds_unc, group):
    """Boolean ROI mask: pixels whose group id is a white mica AND pass both quality gates.

    EMIT reports up to two mineral groups per pixel (group_1 = the ~1 um electronic feature,
    group_2 = the 2.0-2.5 um vibrational feature, where Al-OH lives), so callers check both."""
    ids = ds[f"group_{group}_mineral_id"]
    bd = ds[f"group_{group}_band_depth"]
    fit = ds_unc[f"group_{group}_fit"] * 2.0   # MINUNCERT stores fit at 1/2 scale; undo it here
    want = np.isin(ids.values, wm_ids)                                    # is it a white mica?
    qual = (bd.values >= BAND_DEPTH_MIN) & (fit.values >= FIT_MIN)        # is it trustworthy?
    return want & qual


# Both groups are counted, though after the section 4.3 exclusions every retained entry is a
# group.2um entry, so a non-zero group-1 count would mean the grouping had changed.
for group in (1, 2):
    _m = white_mica_mask(ds_min, ds_minunc, group)
    print(f"group {group}: {int(np.nansum(_m)):,} quality-passing white-mica pixels")

<a id="section-4-5"></a>

### 4.5 Map the white-mica detections

The map that drives the rest of the notebook: where did EMIT detect white mica, and how strongly? We
scan both Tetracorder groups and keep, per pixel, whichever detection has the greater band depth,
then plot that band depth semi-transparently over true-color imagery, so the ridges, drainages and
disturbed areas the alteration follows are visible.

Band depth is a rough expression-strength proxy, not an abundance measurement, and §7 shows how badly
it travels between sensors. Here it serves one purpose: identifying the pixels with a feature strong
enough that measuring its position in section 5 is worth doing.

**Figure 2. EMIT white-mica detections:** band depth of the stronger of the two Tetracorder groups
at every quality-passing pixel, over Esri World Imagery. Hover reads the band depth; the red
outline is the AOI.

In [ ]:
# --- White-mica band depth: better of the two Tetracorder groups per pixel ----------------------
# `lon` / `lat` are the ortho-grid coordinate vectors; every map in this notebook reuses them.
lon = ds_min["longitude"].values
lat = ds_min["latitude"].values

wm_mask = np.zeros((lat.size, lon.size), dtype=bool)
wm_bd = np.zeros((lat.size, lon.size), dtype="float32")
# Either Tetracorder group can carry a white-mica identification. Group 2 (2.0-2.5 µm) is where the
# Al-OH feature lives, so it is evaluated first; group 1 replaces it only where its band depth is
# larger, leaving the stronger of the two expressions at each pixel.
for group in (2, 1):
    m = white_mica_mask(ds_min, ds_minunc, group)
    bd = np.nan_to_num(ds_min[f"group_{group}_band_depth"].values)
    take = m & (bd > wm_bd)
    wm_mask |= m
    wm_bd[take] = bd[take]

n_wm = int(wm_mask.sum())
_frac = 100.0 * n_wm / wm_mask.size
print(f"pixels with a quality-passing white-mica detection : {n_wm:,}  ({_frac:.1f}% of the AOI grid)")
if n_wm:
    print(f"white-mica band depth: median {np.median(wm_bd[wm_mask]):.3f}, "
          f"90th pct {np.percentile(wm_bd[wm_mask], 90):.3f}, max {wm_bd[wm_mask].max():.3f}")

# Wrap the detection rasters as an xarray Dataset on the EMIT ortho grid. hvPlot needs the
# coordinates to place the image geographically; the numpy arrays stay in use by the AVIRIS-5
# comparison, which works in array indices.
wm = xr.Dataset(
    {"band_depth": (("latitude", "longitude"), np.where(wm_mask, wm_bd, np.nan))},
    coords={"latitude": lat, "longitude": lon},
)
wm["band_depth"].attrs["long_name"] = "white-mica Al-OH band depth"

# The 98th percentile, rather than the maximum, sets the top of the color range so that a few very
# deep pixels do not compress the rest of it.
BD_HI = float(np.percentile(wm_bd[wm_mask], 98)) if n_wm else 1.0

detection_map = wm["band_depth"].hvplot.image(
    x="longitude", y="latitude", geo=True, tiles="EsriImagery",
    cmap="Blues", clim=(BAND_DEPTH_MIN, BD_HI), alpha=0.85,
    clabel="white-mica Al-OH band depth",
    title=f"EMIT white-mica detections: {n_wm:,} quality-passing pixels ({sel_name})",
).opts("Image", xrotation=45)

(detection_map * roi_outline).opts(width=780, height=680, padding=0.02)

<a id="section-5"></a>

## 5. Measure the Al-OH absorption center from EMIT reflectance

L2B_MIN has told us where the white micas are. Now we go back to the L2A reflectance, the
underlying measurement, and extract the number that carries the compositional information.

<a id="section-5-1"></a>

### 5.1 Load L2A reflectance with a per-pixel cloud mask

Two details require attention. First, the mask file stores its flag bands in
`sensor_band_parameters/mask_bands` and their order is not guaranteed, so we look the flags up
by name rather than by a hard-coded index, because a fixed index will silently mask the wrong thing on a
file with a different band order. Second, we apply the mask before orthorectification, because
the mask is in raw sensor space and lines up with the unwarped cube.

We also blank out the deep atmospheric-absorption windows using the product's own
`good_wavelengths` flag. Those bands contain no usable surface signal, and leaving them in place
would put spurious minima into any feature search that strayed near 1.4 or 1.9 µm.

> The L2A reflectance granule is the largest transfer in this notebook, roughly 2 GB before
> subsetting, and it is streamed rather than cached. Re-running this cell transfers it again.

In [ ]:
# --- Per-pixel cloud mask, then the reflectance cube it is applied to ----------------------------
# The mask file stores its flag bands in `sensor_band_parameters/mask_bands`, and their order is not
# guaranteed across product versions, so the flags are resolved by name rather than by a fixed
# index: a hard-coded index will silently mask the wrong thing on a file with a different order.
# The buffered cloud mask = Cloud + its Dilated (buffer) + Cirrus. Add "Water flag" or
# "Spacecraft Flag" here if your application needs them.
CLOUD_FLAG_NAMES = ["Cloud flag", "Dilated Cloud Flag", "Cirrus flag"]


def read_cloud_mask(url):
    """Stream the L2A MASK granule and OR-combine the three cloud flags into one raw-space mask."""
    with closing(earthaccess.open([url], provider="LPCLOUD")[0]) as file:
        params = xr.open_dataset(file, engine="h5netcdf", group="sensor_band_parameters")
        mask_bands = [str(b) for b in params["mask_bands"].values]
        params.close()
        print("mask_bands in this file:", mask_bands)
        cloud_flag_idx = [mask_bands.index(n) for n in CLOUD_FLAG_NAMES]
        print("resolved cloud-flag indices:", dict(zip(CLOUD_FLAG_NAMES, cloud_flag_idx)))
        return et.quality_mask(file, cloud_flag_idx)   # OR-combines the flags -> 1 = drop pixel


qmask = read_cloud_mask(mask_url)
n_flagged = int(np.nansum(qmask))
print(f"cloud-masked pixels: {n_flagged:,} of {qmask.size:,} "
      f"({100 * n_flagged / qmask.size:.1f}% of the raw scene)")

# The mask is in raw sensor space, so it has to be applied before orthorectification;
# read_roi_subset opens with ortho=False for exactly that reason, and emit_xarray sets flagged
# pixels to the fill value (-9999).
ds_rfl = read_roi_subset(rfl_url, qmask=qmask)

# Mask fill values (both the -9999 no-data AND the cloud-flagged pixels set to -9999 above) and
# drop bad bands (deep atmospheric windows) via good_wavelengths.
ds_rfl["reflectance"] = ds_rfl["reflectance"].where(ds_rfl["reflectance"] != -9999)
gw = ds_rfl["good_wavelengths"].values.astype(bool)
# The cube is (lat, lon, wavelength), so the band flag applies along the third axis: every pixel of
# a flagged band becomes NaN, which keeps the band axis aligned with the wavelength vector.
ds_rfl["reflectance"].data[:, :, ~gw] = np.nan
wl_emit_um = ds_rfl["wavelengths"].values / 1000.0   # EMIT stores nm; the notebook works in µm
print("EMIT bands:", wl_emit_um.shape, "| range um:",
      round(wl_emit_um.min(), 3), "-", round(wl_emit_um.max(), 3))

<a id="section-5-2"></a>

### 5.2 Continuum removal, step by step

The absorption feature we want sits on top of a broad, sloping background, the continuum, set by
overall albedo, grain size, illumination geometry, and any spectrally-flat darkening from shadow or
coatings. Two pixels of the same mineral can have very different continua while having the same
absorption. Comparing raw reflectance therefore mostly compares brightness.

Continuum removal (Clark & Roush, 1984) fixes this. Over a chosen window we fit a straight line
between the two shoulders and divide the spectrum by it. Both shoulders become exactly 1.0, and
what remains is the absorption on a common baseline: depth becomes comparable between pixels, and
position becomes readable.

What it normalises away: multiplicative brightness differences, and linear spectral slope across
the window. What it does not fix: overlapping features from other minerals inside the window, a
window so narrow it clips the feature's own shoulders, and noise, which continuum removal
amplifies wherever reflectance is low, since we are dividing by a small number.

The cell below shows all of this on the single strongest white-mica pixel in the scene.

**Figure 3. Continuum removal:** the deepest white-mica pixel in the scene, raw and continuum-removed, with the straight line that defines the continuum.

In [ ]:
# --- The deepest white-mica pixel in the scene: raw vs continuum-removed -----------------------
ALOH_WINDOW = (2.10, 2.26)      # the Al-OH search window used throughout this notebook

# Demonstration pixel: the strongest white-mica detection in the scene. Setting every
# non-white-mica pixel to -inf prevents argmax from selecting one; argmax operates on the
# flattened array, so unravel_index converts its single offset back into (row, col).
iy, ix = np.unravel_index(np.argmax(np.where(wm_mask, wm_bd, -np.inf)), wm_bd.shape)
demo_pt = (float(ds_min["latitude"].values[iy]), float(ds_min["longitude"].values[ix]))
print(f"demo pixel: lat {demo_pt[0]:.5f}, lon {demo_pt[1]:.5f}  "
      f"(white-mica band depth {wm_bd[iy, ix]:.3f})")


def emit_spectrum(lat, lon):
    """Nearest-EMIT-pixel reflectance spectrum at (lat, lon): returns (wavelengths_um, reflectance)."""
    p = ds_rfl.sel(latitude=lat, longitude=lon, method="nearest")
    return wl_emit_um, p["reflectance"].values


w_demo, r_demo = emit_spectrum(*demo_pt)
sub_w, sub_r = su.subset_range(w_demo, r_demo, ALOH_WINDOW)
cr_w, cr_r = su.continuum_removed(w_demo, r_demo, wl_range=ALOH_WINDOW)
met_demo = su.aloh_feature_metrics(w_demo, r_demo, wl_range=ALOH_WINDOW)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

# (a) the whole spectrum, with the window marked - context for where we are working.
axes[0].plot(w_demo, r_demo, color="#2c7bb6", lw=1.2)
axes[0].axvspan(*ALOH_WINDOW, color="gold", alpha=0.25)
axes[0].set_title("(a) full EMIT spectrum\n(gold = Al-OH window)")
axes[0].set_xlabel("Wavelength (µm)"); axes[0].set_ylabel("Reflectance")

# (b) the window, with the straight-line continuum we are about to divide by.
finite = np.isfinite(sub_r)
if finite.any():
    # First and last finite samples in the window: argmax on a boolean array returns the first
    # True, and the same call on the reversed array locates the last. Those two points define the
    # straight-line continuum divided out below (Clark & Roush, 1984).
    _i0 = np.argmax(finite); _i1 = len(finite) - 1 - np.argmax(finite[::-1])
    cont = np.interp(sub_w, [sub_w[_i0], sub_w[_i1]], [sub_r[_i0], sub_r[_i1]])
    axes[1].plot(sub_w, sub_r, color="#2c7bb6", lw=1.6, label="reflectance")
    axes[1].plot(sub_w, cont, color="k", ls="--", lw=1.2, label="straight-line continuum")
    axes[1].fill_between(sub_w, sub_r, cont, color="#2c7bb6", alpha=0.15)
axes[1].set_title("(b) the continuum being divided out")
axes[1].set_xlabel("Wavelength (µm)"); axes[1].legend(fontsize=8)

# (c) the result: shoulders at 1.0, and the measurable feature parameters.
axes[2].plot(cr_w, cr_r, color="#2c7bb6", lw=1.8)
axes[2].axhline(1.0, color="k", lw=0.8, alpha=0.5)
if np.isfinite(met_demo["min_wavelength_um"]):
    axes[2].axvline(met_demo["min_wavelength_um"], color="#d7191c", ls=":", lw=1.6,
                    label=f"center {met_demo['min_wavelength_um']:.4f} µm")
    axes[2].annotate("", xy=(met_demo["min_wavelength_um"], 1.0),
                     xytext=(met_demo["min_wavelength_um"], 1.0 - met_demo["band_depth"]),
                     arrowprops=dict(arrowstyle="<->", color="#d7191c", lw=1.2))
    axes[2].text(met_demo["min_wavelength_um"] + 0.004, 1.0 - met_demo["band_depth"] / 2,
                 f"depth\n{met_demo['band_depth']:.3f}", fontsize=8, color="#d7191c")
axes[2].set_title("(c) continuum-removed: position + depth")
axes[2].set_xlabel("Wavelength (µm)"); axes[2].set_ylabel("Continuum-removed reflectance")
axes[2].legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=0.3)
fig.suptitle("Continuum removal on the scene's strongest white-mica pixel", fontweight="bold")
plt.tight_layout(); plt.savefig(FIG_DIR / "02_continuum_removal_demo.png", dpi=130,
                                bbox_inches="tight"); plt.show()

print("\nAl-OH feature metrics for this pixel:")
for k, v in met_demo.items():
    print(f"  {k:22s} {v:.4f}" if isinstance(v, float) else f"  {k:22s} {v}")
print(f"\nA center at {met_demo['min_wavelength_um']:.4f} µm sits in the white-mica range "
      f"(~2.185-2.215 µm);")
print("toward 2.19 implies Al-rich (muscovitic), toward 2.21 implies Fe-Mg-substituted (phengitic).")

<a id="section-5-3"></a>

### 5.3 Absorption-center measurement at every white-mica pixel

The same measurement is now applied at every white-mica pixel. The result is no longer a
classification but a map of a physical quantity, in micrometres, that tracks white-mica chemistry and
therefore the pH of the fluid the mica grew from. The cell below computes that map (`wm_map`),
summarises its distribution and plots the histogram; the map itself is displayed over a small window
at both sensor resolutions in §7.6, and across the full AOI beside the detection map in §7.7.

The color stretch used in §7.6 and §7.7 spans 2.190–2.215 µm, about 25 nm, or roughly 3–4 EMIT bands
at the instrument's 7.4 nm sampling. That is a narrow stretch on a real but small signal, and it is
the reason the sub-pixel question of section 7 matters: a 5 nm artefact from mixing or noise consumes
a fifth of the full dynamic range. The sensitivity analysis of Meyer et al. (2022) puts the
achievable precision in context: for a spectrometer with 8.8 nm sampling and a 9.8 nm bandpass, close
to EMIT's configuration, they report a position RMSE near 1.1 nm at a signal-to-noise ratio of 250
and 1.3 nm at a ratio of 100. Interpret the pattern where it is spatially coherent over many pixels,
not at individual anomalous pixels.

**Figure 4. Distribution of Al-OH absorption centers:** every measured white-mica pixel, with reference lines at 2.190, 2.200 and 2.210 µm.

In [ ]:
# --- Al-OH minimum-wavelength (absorption center) at every white-mica pixel --------------------
refl_cube = ds_rfl["reflectance"].values      # (ny, nx, nbands); cloud-masked in §5.1
# The detection rasters (§4.5) and the reflectance cube must share one ortho grid; every
# per-pixel operation below indexes both with the same (iy, ix).
assert refl_cube.shape[:2] == wm_mask.shape, (
    f"grid mismatch: reflectance {refl_cube.shape[:2]} vs detections {wm_mask.shape}")
# to_micrometers is a no-op on values already in µm; it guards against passing nanometres by mistake.
wl = su.to_micrometers(wl_emit_um)
# Restrict the fit to the Al-OH window. Selecting the spectral axis once here keeps the loop below
# to the handful of bands that carry the feature rather than the full 285-band spectrum.
in_win = (wl >= ALOH_WINDOW[0]) & (wl <= ALOH_WINDOW[1])
print(f"Al-OH window {ALOH_WINDOW[0]}-{ALOH_WINDOW[1]} µm spans {int(in_win.sum())} EMIT bands "
      f"(~{1000 * np.median(np.diff(wl)):.1f} nm sampling)")

# One measurement per white-mica pixel. This explicit loop is the reference implementation; §7.6
# uses the vectorized equivalent (su.aloh_metrics_cube) and cross-checks it against this one.
wm_map = np.full(refl_cube.shape[:2], np.nan)
ys, xs = np.where(wm_mask)
for _iy, _ix in zip(ys, xs):
    spec = refl_cube[_iy, _ix, :]
    if np.isnan(spec[in_win]).all():
        continue
    wm_map[_iy, _ix] = su.minimum_wavelength(wl[in_win], spec[in_win], wl_range=ALOH_WINDOW)

# Carry the measurement on the same grid as the detection rasters, for the map in §7.7.
wm["center"] = (("latitude", "longitude"), wm_map)
wm["center"].attrs["long_name"] = "white-mica Al-OH absorption center (um)"

# Shared colour stretch for every Al-OH position map in this notebook: 2.190 to 2.215 µm spans
# muscovitic to phengitic white mica, so one scale makes EMIT and AVIRIS-5 directly comparable.
# It is defined here, with the measurement, so §7.7 does not depend on having run §7.6.
VMIN, VMAX = 2.190, 2.215

_v = wm_map[np.isfinite(wm_map)]
print(f"\nmeasured centers at {_v.size:,} of {n_wm:,} white-mica pixels")
if _v.size:
    print(f"  range  : {_v.min():.4f} - {_v.max():.4f} µm")
    print(f"  median : {np.median(_v):.4f} µm")
    print(f"  IQR    : {np.percentile(_v, 25):.4f} - {np.percentile(_v, 75):.4f} µm "
          f"({1000 * (np.percentile(_v, 75) - np.percentile(_v, 25)):.1f} nm wide)")

# Check that the grouping built in section 4.3 is spectroscopically clean. Every retained library
# entry places its Al-OH minimum within roughly 2.185-2.215 um, so a measurement near the bottom
# of the search window indicates a failed continuum fit, or a deepest feature that is not Al-OH.
# The mineral id behind any such pixel is named, so the entry responsible is identified directly.
CENTER_FLOOR = 2.15
_low = np.isfinite(wm_map) & (wm_map < CENTER_FLOOR)
print(f"\npixels measured below {CENTER_FLOOR:g} µm: {int(_low.sum()):,}")
for _g in (1, 2):
    _lid = ds_min[f"group_{_g}_mineral_id"].values[_low]
    _lid = _lid[np.isin(_lid, wm_ids)]
    for _i, _n in zip(*np.unique(_lid, return_counts=True)):
        _nm = matrix.loc[matrix["Index"] == int(_i), "Name"]
        print(f"  group {_g}  id {int(_i):3d}  "
              f"{(_nm.iloc[0] if len(_nm) else 'unknown'):42s}  {int(_n):,} px")

# Distribution, with the diagnostic reference positions marked.
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(_v, bins=40, color="#2c7bb6", alpha=0.85, edgecolor="k", linewidth=0.3)
for x, lbl, c in [(2.190, "Al-rich muscovite", "#1a9850"),
                  (2.200, "muscovite / illite", "#666666"),
                  (2.210, "phengitic (Fe-Mg)", "#d73027")]:
    ax.axvline(x, color=c, ls="--", lw=1.3)
    ax.text(x, ax.get_ylim()[1] * 0.95, f" {lbl}", rotation=90, va="top", fontsize=8, color=c)
ax.set_xlabel("Al-OH absorption center (µm)")
ax.set_ylabel("EMIT pixels")
ax.set_title("Distribution of white-mica Al-OH centers across the ROI")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR / "02_center_histogram.png", dpi=130,
                                bbox_inches="tight"); plt.show()

<a id="section-6"></a>

## 6. Search and access AVIRIS-5 airborne reflectance

Now we drop to the airborne scale. AVIRIS-5 (ORNL DAAC) provides orthocorrected surface
reflectance. Because AVIRIS-5 flies on an aircraft, its ground sampling distance (GSD) is a
function of how high the plane is flying - across campaigns it ranges from roughly 0.5 m to
15 m. We will read the GSD of the specific flight line we download straight from its grid
coordinates, in the access cell below, rather than assuming a value.

| Product | short_name / concept-id |
|---------|-------------------------|
| AVIRIS-5 L2A Ortho Surface Reflectance | `AV5_L2A_RFL_2484` · `C4184090548-ORNL_CLOUD` · DOI 10.3334/ORNLDAAC/2484 |

> Note: the AVIRIS-5 L1B radiance collection is `C4076410273-ORNL_CLOUD`; for the spectral
> comparison here we need the L2A reflectance collection above.

<a id="section-6-1"></a>

### 6.1 Search for flight lines over the study area

AVIRIS-5 L2A orthorectified surface reflectance is distributed by the ORNL DAAC, one granule per
flight line. The same `earthaccess` search pattern applies; only the collection changes.

In [ ]:
# --- Search NASA CMR for AVIRIS-5 L2A reflectance granules over the ROI -------------------------
# AVIRIS-5 is the airborne counterpart to EMIT: the same shortwave-infrared spectroscopy, flown on an
# aircraft at ~10 m ground sampling instead of ~60 m from orbit. Each granule is one flight line. We
# query the ORNL DAAC AVIRIS-5 L2A reflectance collection by concept-id + ROI, exactly as we did for
# EMIT, so the rest of the notebook can compare the two sensors over the same ground.
AVIRIS5_L2A_RFL_CID = "C4184090548-ORNL_CLOUD"

aviris_results = earthaccess.search_data(
    concept_id=AVIRIS5_L2A_RFL_CID,   # AVIRIS-5 L2A reflectance, ORNL cloud collection
    polygon=roi_coords,               # same ROI polygon as the EMIT search
    count=500,
)
print(f"AVIRIS-5 L2A RFL granules over ROI: {len(aviris_results)}")
# The airborne section cannot run without at least one line; warn early (rather than failing deep in
# a later cell) if the collection/ROI returns nothing.
if len(aviris_results) == 0:
    print("\n*** No AVIRIS-5 reflectance granules returned for this ROI/time. ***")
    print("The airborne section below requires at least one granule; re-check the ROI or the")
    print("collection concept-id before proceeding.")

In [ ]:
# --- Tabulate the available flight lines (name + acquisition time) ------------------------------
# Print the search result, which lists the flight lines available and when they were flown,
# before the next cell pins the specific line this tutorial reads. Two small accessors read the
# granule name and start time from the CMR metadata (with a fallback for either metadata shape).
def aviris_name(r):
    try:
        return r["umm"]["GranuleUR"]              # the flight-line granule id, e.g. AV520250723t163747_010_...
    except Exception:
        return r["meta"]["native-id"]

def aviris_time(r):
    try:
        return r["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]   # ISO acquisition start
    except Exception:
        return None

av_table = pd.DataFrame(
    {"granule": [aviris_name(r) for r in aviris_results],
     "time": [aviris_time(r) for r in aviris_results]}
)
print(av_table.head(15).to_string())   # preview the first 15 lines (there may be more)

<a id="section-6-2"></a>

### 6.2 Select one flight line

A flight line is a narrow diagonal swath, and the CMR footprint is a coarse bounding quadrilateral
whose corners cover ground the sensor never imaged. One line is pinned by name below, and its
footprint is then checked against the AOI.

We pin one flight line, `AV520250723t163747_010_L2A_RFL_1`, and read the AOI window out of it. The
AVIRIS-5 swath is narrow, imaging a diagonal corridor roughly 22 km across, but the White Mica AOI
lies inside that corridor: one line therefore covers the full AOI and all three comparison sites of
§7.1. If the pinned granule is ever missing (the collection is re-versioned, the AOI is moved),
the code falls back to the first line whose footprint contains the strongest white-mica pixel of
§5.2, and finally to the first line returned. We then download that line's reflectance netCDF.
AVIRIS-5 reflectance is a NetCDF datatree: the cube is at `['reflectance']['reflectance']` with
`easting`/`northing` map coordinates and a `wavelength` (singular) coordinate.

In [ ]:
def aviris_footprint(r):
    try:
        g = r["umm"]["SpatialExtent"]["HorizontalSpatialDomain"]["Geometry"]
        gp = g["GPolygons"][0]["Boundary"]["Points"]
        return shape({"type": "Polygon",
                      "coordinates": [[(p["Longitude"], p["Latitude"]) for p in gp]]})
    except Exception:
        return None

# --- Pin the flight line read by this tutorial -------------------------------------------------
# One line is sufficient for this AOI: the corridor imaged by this acquisition spans the whole ROI,
# including the three comparison sites of §7.1. The match is on the granule-name stem (prefix),
# so a product-version suffix on the GranuleUR still matches.
AVIRIS_LINE_ID = "AV520250723t163747_010_L2A_RFL_1"

sel_av = next((r for r in aviris_results if aviris_name(r).startswith(AVIRIS_LINE_ID)), None)
if sel_av is None:
    print(f"WARNING: pinned line {AVIRIS_LINE_ID!r} not found in the AVIRIS-5 search results.")

# If the pinned line is absent (collection re-versioned, AOI moved, and so on), fall back to a
# geometry rule, the first line whose footprint contains the target pixel, and then to the first
# line returned, so the airborne section still has data to run on.
if sel_av is None:
    print("Pinned line unavailable; falling back to footprint-containment selection.")
    target = Point(demo_pt[1], demo_pt[0]) if demo_pt else roi_poly.centroid
    sel_av = next((r for r in aviris_results
                   if aviris_footprint(r) is not None
                   and aviris_footprint(r).contains(target)), None)
    if sel_av is None and len(aviris_results):
        sel_av = aviris_results[0]

print("selected AVIRIS-5 line:", aviris_name(sel_av) if sel_av is not None else None)

# Report the temporal separation that actually matters for the comparison: EMIT scene date vs the date
# of THIS flight line (the granule id encodes it as AV5 + YYYYMMDD). Section 3 ranked EMIT scenes
# against the campaign's median flight date, which is a season anchor rather than this line's date.
_m_av = _re.search(r"(\d{8})[tT]\d{6}", aviris_name(sel_av)) if sel_av is not None else None
if _m_av and _sel_dt is not None:
    av_line_date = _dt.datetime.strptime(_m_av.group(1), "%Y%m%d")
    print(f"  flown {av_line_date.date()}  |  EMIT scene {_sel_dt.date()}  ->  "
          f"{abs((_sel_dt - av_line_date).days)} days between the two acquisitions")

<a id="section-6-3"></a>

### 6.3 Cache the flight line, with an integrity check

The reflectance files here are gigabytes, so downloads are cached. A download interrupted partway,
whether by a dropped connection or the kernel being killed under memory pressure, leaves a truncated netCDF on
disk, and trusting `size > 0` alone means every later run reuses that corrupt file and fails on open,
with an error that points at the file rather than at the download. So a cached file is verified to
open as HDF5 before it is reused.

In [ ]:
# An interrupted transfer leaves a plausible-looking file on disk. The cheapest way to detect one
# is to try to open it as HDF5, the container format that netCDF-4 uses.
def _is_readable_netcdf(path):
    """True if `path` opens as HDF5/netCDF-4 (catches truncated/partial downloads cheaply)."""
    try:
        with h5py.File(path, "r"):
            return True
    except Exception:
        return False


def cached_download(url, subdir=DATA_DIR):
    """Download `url` to `subdir` unless a complete, readable copy is present; return local path."""
    dst = Path(subdir) / url.split("/")[-1]      # keep the server-side filename
    # Three cases to distinguish: a complete local copy (reuse it), a partial local copy (discard
    # it and fetch again), or nothing local yet (fetch it).
    if dst.exists() and dst.stat().st_size > 0:
        if _is_readable_netcdf(dst):
            print("cached:", dst.name)
            return dst
        print("cached file is truncated/corrupt, re-downloading:", dst.name)
        dst.unlink()
    else:
        print("downloading:", dst.name)
    earthaccess.download([url], local_path=str(subdir))
    # Verify the transfer as well as the cache: a truncated file would otherwise fail much later,
    # inside an xarray call, where the cause is considerably harder to recognise.
    if not _is_readable_netcdf(dst):
        raise OSError(f"download did not produce a readable netCDF: {dst.name} "
                      f"({dst.stat().st_size if dst.exists() else 0} bytes) - re-run to retry.")
    return dst

The cell below downloads that line and opens it as a lazily-read reflectance cube.

> A flight line is a narrow diagonal strip, but the granule footprint polygon is a rough bounding
> quadrilateral, so its corners cover ground the sensor never imaged. Footprint containment is
> therefore not sufficient: the site selection in §7 also requires real reflectance under the pixel.

The reflectance cube is addressed through a GDAL VRT, the GDAL Virtual Format. A VRT is an XML file
that describes a dataset rather than storing one: it records the raster grid size, the affine
geotransform and the CRS, and for each band a pointer to a band of a source dataset, here the NetCDF
reflectance variable, with the AVIRIS fill value of -9999 declared as nodata. No pixel data is copied
into the descriptor. GDAL serves every read from the referenced source on demand, so a windowed read
of the ~2.2 GB airborne cube touches only the blocks that window covers and the whole cube is never
held in memory. The GSD is read from the raster grid rather than assumed.

In [ ]:
def aviris_urls(r):
    try:
        return r.data_links()
    except Exception:
        return [u["URL"] for u in r["umm"]["RelatedUrls"] if u.get("Type") == "GET DATA"]

# Download the reflectance netCDF for the selected line (one file per granule). `cached_download`
# reuses a complete local copy, so this is a no-op on a second run.
_av_url = [u for u in aviris_urls(sel_av) if u.endswith(".nc") and ("RFL" in u.upper())][0]
print("downloading:", _av_url.split("/")[-1])
fp_av = cached_download(_av_url)

In [ ]:
# =====================================================================================================
# Open the AVIRIS-5 flight line lazily, through GDAL, with the fill value declared.
# =====================================================================================================
# The reflectance file is ~2.2 GB (424 wavelength bands x ~1400 x ~2100 pixels), so nothing is read
# eagerly. GDAL addresses a single group/variable inside a NetCDF with the NETCDF:"file":/group/var
# syntax; wrapping that subdataset in a VRT descriptor (an XML file that points at the source bands
# instead of holding pixels) gives a plain 424-band raster with an affine transform, a CRS and a
# declared nodata value, which is what `rasterio` windowed reads and `rioxarray` masking both want.
# GDAL resolves each read against the NetCDF on demand, so nothing is copied or cached. This is
# the same rasterio/GDAL access pattern used in the ORNL DAAC AVIRIS reference notebooks.


# Build a cache key from the selected line name, so that a different choice of line writes a
# different cache file and a descriptor built from another line is never silently reused. The
# granule's line-number token (e.g. "010") keeps the filename short and stable.
def _line_token(name):
    # granule names look like AV520250723t163747_010_L2A_RFL_1 -> grab the "010" field
    parts = name.split("_")
    return parts[1] if len(parts) > 1 else name
_av_key = _line_token(aviris_name(sel_av))                   # e.g. "010"
AV_VRT = DATA_DIR / f"aviris5_{_av_key}.vrt"                 # cached virtual-raster descriptor

# The reflectance cube lives at /reflectance/reflectance in every AVIRIS-5 L2A file.
av_subdataset = f'NETCDF:"{fp_av}":/reflectance/reflectance'

# --- Build (or reuse) the VRT -----------------------------------------------------------------------
# Caching rule: only (re)build the VRT if it does not already exist, so re-running the notebook is
# instant. Because the filename carries the line token, a different line writes its own VRT rather
# than overwriting this one.
if AV_VRT.exists():
    print("cached VRT:", AV_VRT.name)
else:
    print("building VRT for:", av_subdataset.split('"')[1].split("/")[-1])
    # srcNodata/VRTNodata = -9999 (the AVIRIS fill value): declaring it here is what lets rioxarray
    # mask fill pixels to NaN, and keeps them out of every statistic computed below.
    gdal.BuildVRT(str(AV_VRT), [av_subdataset], srcNodata=-9999, VRTNodata=-9999).FlushCache()
    print("wrote", AV_VRT.name, f"({AV_VRT.stat().st_size/1e3:.0f} KB)")

# --- The wavelength axis (VRT bands are just numbered 1..424; recover the actual wavelengths) --------
# GDAL sees the 424 spectral bands as anonymous raster bands, so we read the real per-band wavelengths
# straight from the source file.
av_wl_nm = xr.open_datatree(str(fp_av), engine="h5netcdf")\
             .reflectance.to_dataset()["reflectance"]["wavelength"].values

# --- Open the cube lazily ---------------------------------------------------------------------------
# masked=True applies the -9999 nodata as NaN; chunks=... keeps it lazy (dask) so nothing is read until
# a .sel()/.values below actually needs those pixels. rioxarray names the dims band/y/x; we rename to
# the AVIRIS conventions (wavelength/northing/easting) the rest of the notebook already expects.
av_rfl = (rxr.open_rasterio(str(AV_VRT), masked=True, chunks={"band": -1, "x": 512, "y": 512})
          .rename({"band": "wavelength", "x": "easting", "y": "northing"})
          .assign_coords(wavelength=av_wl_nm))
av_crs_wkt = av_rfl.rio.crs.to_wkt()          # UTM CRS carried through the VRT from the source file

# --- Trim the lazy cube to the ROI (+small pad) ------------------------------------------------------
# The flight line images a corridor roughly 22 km long, while the ROI is a small slice of that.
# Clipping the (still lazy) cube to the ROI bounding box in the AVIRIS UTM CRS means the one-time
# ROI RGB read of §6.5 and every downstream slice work over the ROI only, not over
# the whole flight corridor. This is a coordinate slice (no data are read yet); nodata beyond the
# line's edge stays NaN.
_roi_utm = roi_gdf.to_crs(av_crs_wkt).total_bounds        # (minx, miny, maxx, maxy) in metres
_pad = 300.0                                              # metres of margin around the ROI
_desc = av_rfl["northing"].values[0] > av_rfl["northing"].values[-1]   # northing descending?
av_rfl = av_rfl.sel(
    easting=slice(_roi_utm[0] - _pad, _roi_utm[2] + _pad),
    northing=(slice(_roi_utm[3] + _pad, _roi_utm[1] - _pad) if _desc
              else slice(_roi_utm[1] - _pad, _roi_utm[3] + _pad)))     # hi->lo when descending

av_wl_um = su.to_micrometers(av_rfl["wavelength"].values)

# rioxarray returns northing DESCENDING (row 0 = north), which is exactly what the origin="upper" RGB
# of section 7.3 and the descending-northing slice logic there assume, so no re-sorting is needed.
print(f"AVIRIS-5 cube (lazy) -> (wavelength, northing, easting) = "
      f"{tuple(av_rfl.sizes[d] for d in ('wavelength', 'northing', 'easting'))}")
print("AVIRIS-5 bands:", av_wl_um.shape,
      "| range um:", round(float(av_wl_um.min()), 3), "-", round(float(av_wl_um.max()), 3))
print("north descending:", bool(av_rfl["northing"].values[0] > av_rfl["northing"].values[-1]))

# Short label for figure titles.
aviris_line_name = aviris_name(sel_av)

# --- Read the ground sampling distance (GSD) from the grid coordinates ------------------------------
# AVIRIS-5 is airborne, so the GSD depends on how high the plane was flying (typically ~0.5-15 m across
# campaigns). Rather than assume a value, we measure it: easting/northing are in metres (metre-based
# UTM CRS), so the median coordinate step IS the pixel size.
AV_GSD_X = float(np.median(np.abs(np.diff(av_rfl["easting"].values))))
AV_GSD_Y = float(np.median(np.abs(np.diff(av_rfl["northing"].values))))
AV_GSD = round(0.5 * (AV_GSD_X + AV_GSD_Y), 2)    # ~identical in x and y; use the mean for labels
print(f"AVIRIS-5 GSD for this flight line: {AV_GSD_X:.2f} m (easting) x {AV_GSD_Y:.2f} m (northing) "
      f"-> {AV_GSD:.1f} m")
print("  (airborne GSD is set by flight altitude; AVIRIS-5 is generally ~0.5-15 m per campaign.)")

<a id="section-6-4"></a>

### 6.4 How pixels are read from the cube, and why the access pattern matters

`av_rfl` above is a lazy `xarray` view of the flight line, and from here on we use it only for its
coordinate and CRS metadata. Every read of pixel data goes through a small `av_read()` helper backed
by a direct `rasterio` window. This is because the two access patterns differ in cost by roughly
three orders of magnitude.

This access difference is because AVIRIS-5 reflectance is stored band-sequentially in 256 × 256
compressed tiles. Each of the 424 wavelengths is its own image plane. So the cost of a read is set by
how many (band, tile) pairs it touches:

| read | tiles touched | cost |
|---|---|---|
| a few bands over a wide area | tens per band | inexpensive; sufficient for mapping |
| all 424 bands at one pixel | 424 (one per band) ≈ 110 MB decompressed | expensive |

A lazily-chunked `xarray`/`dask` cube inverts that. With `chunks={"band": -1, ...}` every chunk spans
all 424 bands, so asking for one pixel's spectrum materialises a ~440 MB chunk, and dask does not
cache between `.compute()` calls, so the next pixel pays it again. Selecting three bands through such
a chunked array reads all 424 anyway.

That is the pattern the site search in §7 needs (probe many candidate pixels), so we restructure it:
one cheap multi-band read up front produces a reusable validity raster, and the per-site
full-spectrum reads happen once per site and are cached. The results are identical, while the runtime
falls from tens of minutes to seconds.

In [ ]:
# --- A windowed reader for the AVIRIS-5 flight line ----------------------------------------------
# One persistent GDAL handle on the VRT, plus the pixel window corresponding to the ROI-clipped
# extent of `av_rfl`. Deriving the window from av_rfl's own coordinates (rather than re-deriving it
# from bounds) guarantees that row/column indices into anything av_read() returns line up exactly
# with av_north / av_east, so an index computed from a lat/lon is valid for every array below.
av_src = rasterio.open(str(AV_VRT))
av_east = av_rfl["easting"].values          # ROI grid, ascending
av_north = av_rfl["northing"].values        # ROI grid, descending: row 0 is the north edge
AV_NY, AV_NX = av_north.size, av_east.size

_av_proj_read = Proj(av_crs_wkt)            # lon/lat -> AVIRIS UTM (vectorized)

_tf = av_src.transform                      # full-line affine (a = +pixel size, e = -pixel size)
# Convert the ROI's first pixel centre into a whole-raster pixel index. The affine maps pixel
# corners, with (c, f) the coordinates of the raster's upper-left corner, so subtracting 0.5
# converts a corner-referenced offset into a centre-referenced one before rounding. Window takes
# (col_off, row_off, width, height), column first, unlike the (row, col) order used elsewhere.
AV_WIN = Window(int(round((av_east[0] - _tf.c) / _tf.a - 0.5)),
                int(round((av_north[0] - _tf.f) / _tf.e - 0.5)), AV_NX, AV_NY)
print(f"ROI window in the flight line: {AV_NY} rows x {AV_NX} cols at "
      f"(row {AV_WIN.row_off}, col {AV_WIN.col_off}) of {av_src.height} x {av_src.width}")
print(f"source tiling: {av_src.block_shapes[0][0]} x {av_src.block_shapes[0][1]} per band, "
      f"{av_src.count} bands")


def av_read(bands=None, rows=None, cols=None):
    """Read AVIRIS-5 reflectance over a sub-block of the ROI grid.

    bands : 1-based band numbers (None = all 424). rows/cols : (start, stop) index pairs into the
    ROI grid, i.e. into av_north / av_east (None = the full ROI extent).
    Returns float64 (n_bands, n_rows, n_cols) with fill / non-positive values as NaN."""
    r0, r1 = rows if rows is not None else (0, AV_NY)
    c0, c1 = cols if cols is not None else (0, AV_NX)
    win = Window(AV_WIN.col_off + c0, AV_WIN.row_off + r0, c1 - c0, r1 - r0)
    a = av_src.read(indexes=(None if bands is None else [int(b) for b in bands]),
                    window=win).astype("float64")
    a[a <= 0] = np.nan                      # -9999 fill and any non-physical reflectance
    return a


def av_rowcol(lat, lon):
    """Nearest ROI row/col for (lat, lon); scalars or arrays accepted. Out-of-grid values are not clipped."""
    x, y = _av_proj_read(np.asarray(lon, dtype="float64"), np.asarray(lat, dtype="float64"))
    # Distance from the grid origin in metres, divided by the pixel size and rounded, gives the
    # row/column whose centre is nearest. Northing decreases with row, so that difference is taken
    # in the opposite sense from the easting difference.
    return (np.rint((av_north[0] - y) / AV_GSD_Y).astype("int64"),
            np.rint((x - av_east[0]) / AV_GSD_X).astype("int64"))


def av_index_box(e_lo, e_hi, n_lo, n_hi, tol=1e-6):
    """(rows, cols) half-open index ranges for the pixels whose CENTRES fall in an
    easting/northing box, the same selection xarray's .sel(slice(...)) makes, clipped to the
    ROI grid. Northing is descending, so a northward box maps to LOWER row indices."""
    # Ceiling on the low edge and floor on the high edge keep only pixels whose centres fall
    # inside the box; `tol` absorbs floating-point noise so a centre lying exactly on an edge is
    # not lost; the +1 turns an inclusive last index into a half-open Python stop value.
    c0 = int(np.ceil((e_lo - av_east[0]) / AV_GSD_X - tol))
    c1 = int(np.floor((e_hi - av_east[0]) / AV_GSD_X + tol)) + 1
    r0 = int(np.ceil((av_north[0] - n_hi) / AV_GSD_Y - tol))
    r1 = int(np.floor((av_north[0] - n_lo) / AV_GSD_Y + tol)) + 1
    return ((min(max(r0, 0), AV_NY), min(max(r1, 0), AV_NY)),
            (min(max(c0, 0), AV_NX), min(max(c1, 0), AV_NX)))


# Band mask for the atmospheric-absorption windows and spectral tails, precomputed once so the
# per-site reads below can blank them with a single boolean index instead of four comparisons.
_AV_BAD = np.zeros(av_wl_um.size, dtype=bool)
# The four intervals are, in order: the blue end below 0.45 um where signal-to-noise is poor, the
# 1.4 um and 1.9 um atmospheric water-vapour absorptions, and the long-wavelength detector tail
# beyond 2.46 um. The 99.0 upper bound is an open-ended sentinel above the spectral range.
for _lo, _hi in [(0.0, 0.45), (1.34, 1.48), (1.80, 1.98), (2.46, 99.0)]:
    _AV_BAD |= (av_wl_um >= _lo) & (av_wl_um <= _hi)
print(f"bands blanked as atmospheric / spectral-tail: {int(_AV_BAD.sum())} of {av_wl_um.size}")

<a id="section-6-5"></a>

### 6.5 One up-front read for display and validity

One up-front read pulls a handful of bands over the whole ROI and gets us two things we need
repeatedly:

- `av_rgb_raw`: the three visible bands, for the true-color panel in §7.3.
- `av_valid`: a boolean "this pixel has real airborne data" raster, sampled inside the Al-OH
  window rather than across the whole spectrum. A candidate site is only useful if there is real
  reflectance where the measurement is made, not merely somewhere in the visible.

A flight line is a narrow diagonal swath, but its metadata footprint is a rough bounding
quadrilateral whose corners cover ground the sensor never imaged. A point can sit inside the polygon
and have no spectral information. §7 tests every candidate against this raster with an array lookup,
without reading from disk.

In [ ]:
# --- One up-front multi-band ROI read: display RGB + a validity raster --------------------------
# Caching: this is the only wide-area read in the notebook. It is keyed on the flight-line number
# and shape-checked on load, so a different line selection or ROI rebuilds it rather than
# silently reusing the wrong grid.
def _nearest_band(wl_um, target_um):
    """Index of the band whose wavelength is closest to target_um (bands are not on round nm)."""
    return int(np.nanargmin(np.abs(np.asarray(wl_um) - target_um)))


RGB_UM = (0.650, 0.560, 0.470)     # R, G, B target wavelengths (um) ~ human-visible red/green/blue
av_rgb_idx = [_nearest_band(av_wl_um, t) for t in RGB_UM]
# 4 bands spread across the Al-OH window is plenty to tell real reflectance from fill, and keeps
# this read ~7 bands rather than 424.
# np.where returns the band indices inside the Al-OH window, [[0, -1]] keeps the first and last
# of them, and np.linspace(start, stop, 4) spreads four evenly spaced indices across that span. The
# leading * unpacks those two ends as linspace's start and stop.
AV_VALID_IDX = np.linspace(*np.where((av_wl_um >= ALOH_WINDOW[0])
                                     & (av_wl_um <= ALOH_WINDOW[1]))[0][[0, -1]], 4).astype(int)

AV_ROI_CACHE = DATA_DIR / f"aviris5_roi_cache_{_av_key}.npz"
_cached = None
if AV_ROI_CACHE.exists():
    _c = np.load(AV_ROI_CACHE)
    if _c["rgb"].shape == (AV_NY, AV_NX, 3) and _c["valid"].shape == (AV_NY, AV_NX):
        _cached = _c
        print("cached ROI read:", AV_ROI_CACHE.name)
    else:
        print("cached ROI read has the wrong grid, rebuilding")

if _cached is not None:
    av_rgb_raw, av_valid = _cached["rgb"].astype("float64"), _cached["valid"]
else:
    print(f"reading {len(av_rgb_idx) + AV_VALID_IDX.size} bands over the "
          f"{AV_NY} x {AV_NX} ROI (one time, then cached) ...")
    _t0 = time.time()
    # A single read covering all seven bands. The +1 converts 0-based numpy indices into the
    # 1-based band numbers GDAL, and therefore rasterio, expects.
    _blk = av_read(bands=np.concatenate([np.array(av_rgb_idx), AV_VALID_IDX]) + 1)
    av_rgb_raw = np.transpose(_blk[:3], (1, 2, 0))                 # -> (northing, easting, band)
    av_valid = np.isfinite(_blk[3:]).mean(axis=0) >= 0.75          # >= 3 of the 4 probe bands real
    np.savez_compressed(AV_ROI_CACHE, rgb=av_rgb_raw.astype("float32"), valid=av_valid,
                        east=av_east, north=av_north)
    print(f"  done in {time.time() - _t0:.1f} s -> {AV_ROI_CACHE.name} "
          f"({AV_ROI_CACHE.stat().st_size / 1e6:.0f} MB)")

print(f"AVIRIS-5 pixels with real data in the Al-OH window: {int(av_valid.sum()):,} of "
      f"{av_valid.size:,} ({100 * av_valid.mean():.0f}% of the ROI window)")
print("  (the rest is fill outside the diagonal swath; see the RGB panel in section 7.3)")

<a id="section-7"></a>

## 7. Compare EMIT and AVIRIS-5 across scales

The comparison runs from three sites to a map. First choose sites that span the compositional range
and are backed by real airborne data (§7.1), quantify how many airborne pixels fall inside one EMIT
pixel (§7.2), and locate the sites in true color at both scales (§7.3). Then examine the raw spectra
(§7.4), continuum-remove them and measure the feature (§7.5), and finally extend the same
measurement from three sites to a map (§7.6) and to the full scene (§7.7).

<a id="section-7-1"></a>

### 7.1 Choose comparison sites that span the composition range

We now have an EMIT absorption-center map (§5.3) and an AVIRIS-5 reflectance cube (§6). To compare
them we need specific ground locations, and the choice of locations determines what the comparison
can show.

The single deepest white-mica pixel would be uninformative, because it samples one point of the
compositional distribution. The selection below instead takes three white-mica pixels that span
the observed Al-OH range: one at the short-wavelength (Al-rich, muscovitic) end, one near the
middle, and one at the long-wavelength (Fe–Mg-substituted, phengitic) end. Agreement at all three
establishes that the gradient in the §5.3 map is real, and the middle site tests whether agreement
at the extremes was coincidental.

Three constraints on every candidate:

1. it must be a quality-passing detection (the §4.4 gates);
2. it must fall inside the AVIRIS-5 swath footprint; and
3. the nearest AVIRIS pixel must carry real reflectance, not fill: a flight line is a narrow
   diagonal swath and its bounding polygon contains large empty corners, so being "inside the
   footprint" is not the same as having data.

The band depth is also required to be in the upper half of the distribution. An absorption center
measured on a shallow feature is dominated by noise, and choosing the extremes of a noisy quantity
selects artefacts rather than geology.

Constraint 3 is where a direct implementation becomes too slow for a live session. Tested one
candidate at a time, it costs a full 424-band point read per candidate, which §6.4 showed to be the
most expensive operation on these files, so screening a few hundred candidates that way takes tens
of minutes. Tested against the `av_valid` raster from §6.5 it is a vectorized array lookup over
every qualifying pixel at once, with no disk access. The criterion is identical and the result
returns immediately.

In [ ]:
# --- Where does the AVIRIS-5 flight line actually have data? ------------------------------------
# The granule footprint is a first, inexpensive filter: a candidate site outside it cannot hold
# airborne data. It is necessary but not sufficient, as aviris_has_data() below makes precise.
av_fp = aviris_footprint(sel_av)

_av_proj_sel = Proj(av_crs_wkt)             # lat/lon <-> AVIRIS projected metres (as in §6.4)


def aviris_has_data(lat, lon):
    """Do the nearest AVIRIS pixels to (lat, lon) carry real reflectance rather than fill?

    Vectorized over arrays of lat/lon, and a pure lookup into the `av_valid` raster built once in
    §6.4, with no disk access. Points outside the ROI grid come back False."""
    iy, ix = av_rowcol(lat, lon)
    inside = (iy >= 0) & (iy < AV_NY) & (ix >= 0) & (ix < AV_NX)
    out = np.zeros(inside.shape, dtype=bool)
    out[inside] = av_valid[iy[inside], ix[inside]]
    return out


def aviris_valid_candidates(mask, score):
    """Every pixel in `mask` with valid AVIRIS data, ranked by `score` descending.

    Returns a list of (iy, ix, lat, lon). Because the AVIRIS validity test is an array lookup
    (§6.5), every qualifying pixel can be screened rather than only the deepest few hundred."""
    iy, ix = np.where(mask & np.isfinite(score))
    if iy.size == 0:
        return []
    la = ds_min["latitude"].values[iy]
    lo = ds_min["longitude"].values[ix]
    keep = aviris_has_data(la, lo)                  # constraint 3, all candidates at once
    iy, ix, la, lo = iy[keep], ix[keep], la[keep], lo[keep]
    if av_fp is not None and iy.size:
        # Constraint 2. Point-in-polygon is a cheap Python loop now that only pixels which already
        # have real data reach it (the raster test is the strictly stronger of the two).
        keep = np.fromiter((av_fp.contains(Point(a, b)) for a, b in zip(lo, la)),
                           dtype=bool, count=iy.size)
        iy, ix, la, lo = iy[keep], ix[keep], la[keep], lo[keep]
    order = np.argsort(score[iy, ix])[::-1]
    return [(int(iy[k]), int(ix[k]), float(la[k]), float(lo[k])) for k in order]


# --- Three white micas spanning the Al-OH range, among well-expressed features only -------------
_wm_ok = wm_mask & np.isfinite(wm_map)
_depth_gate = float(np.nanpercentile(wm_bd[_wm_ok], 50)) if _wm_ok.any() else BAND_DEPTH_MIN
_wm_strong = _wm_ok & (wm_bd >= _depth_gate)
print(f"white-mica pixels with a measured center : {int(_wm_ok.sum()):,}")
print(f"  ... and band depth >= {_depth_gate:.3f} (median) : {int(_wm_strong.sum()):,}")

_t0 = time.time()
wm_cands = aviris_valid_candidates(_wm_strong, wm_bd)
print(f"  ... and inside the AVIRIS swath with valid data : {len(wm_cands):,} "
      f"(all candidates screened, {time.time() - _t0:.2f} s)")

wm_short = wm_mid = wm_long = None
if wm_cands:
    # Rank the surviving candidates by Al-OH center and take the two ends plus the median. Using the
    # median rather than the arithmetic midpoint keeps the middle site on a pixel that actually
    # exists, and keeps it away from any thin tail in the position distribution.
    _pos = np.array([wm_map[iy, ix] for iy, ix, _, _ in wm_cands])
    _order = np.argsort(_pos)
    wm_short = wm_cands[int(_order[0])]                      # Al-rich / muscovitic end
    wm_mid = wm_cands[int(_order[len(_order) // 2])]         # middle of the observed range
    wm_long = wm_cands[int(_order[-1])]                      # Fe-Mg-substituted / phengitic end
    print(f"\n  Al-rich end   : center {_pos[_order[0]]:.4f} µm at "
          f"({wm_short[2]:.5f}, {wm_short[3]:.5f})")
    print(f"  mid-range     : center {_pos[_order[len(_order) // 2]]:.4f} µm at "
          f"({wm_mid[2]:.5f}, {wm_mid[3]:.5f})")
    print(f"  Fe-Mg end     : center {_pos[_order[-1]]:.4f} µm at "
          f"({wm_long[2]:.5f}, {wm_long[3]:.5f})")
    print(f"  end-to-end    : {1000 * (_pos.max() - _pos.min()):.1f} nm "
          f"({(_pos.max() - _pos.min()) / np.median(np.diff(wl)):.1f} EMIT bands)")

# --- The comparison-site list used by every figure from here on ---------------------------------
# (label, (lat, lon), marker, color). Markers and colors are fixed so the same site is visually
# identifiable across the RGB map, the spectra, and the proxy map. The colors run cool -> warm with
# increasing Al-OH wavelength, matching the RdYlBu_r position maps of sections 7.6 and 7.7.
SITES = [(lbl, (c[2], c[3]), mk, cl) for lbl, c, mk, cl in [
    ("White mica, Al-rich", wm_short, "s", "#2c7bb6"),
    ("White mica, mid-range", wm_mid, "o", "#fdae61"),
    ("White mica, Fe-Mg", wm_long, "^", "#762a83"),
] if c is not None]
# Guard the degenerate case of very few candidates, where two picks can land on the same pixel.
_seen = set()
SITES = [s for s in SITES if not (s[1] in _seen or _seen.add(s[1]))]

print(f"\n{len(SITES)} comparison sites:")
for lbl, pt, _mk, _cl in SITES:
    print(f"  {lbl:22s} lat {pt[0]:.5f}, lon {pt[1]:.5f}")
if not SITES:
    raise RuntimeError(
        "No site satisfies all three constraints (a quality-passing white-mica detection, a "
        "measurable Al-OH center, and valid AVIRIS-5 reflectance). Every figure in section 7 "
        "needs at least one site, so stop here: relax BAND_DEPTH_MIN / FIT_MIN, or widen the AOI.")
if len(SITES) < 3:
    print("\nWARNING: fewer than 3 sites found. The AVIRIS swath may barely overlap the white-mica")
    print("alteration in this scene - relax BAND_DEPTH_MIN / FIT_MIN, or widen the AOI.")

<a id="section-7-2"></a>

### 7.2 The number of AVIRIS pixels within one EMIT pixel

The cell below computes the EMIT pixel footprint in metres from the ortho-grid spacing (rather than
assuming 60 m), divides by the AVIRIS GSD read from the flight-line grid in §6, and reports how many
airborne pixels each site's EMIT footprint contains.

The gather function it defines is the primary data-access routine for the sections that follow: given
a site, it returns every valid AVIRIS spectrum inside the corresponding EMIT pixel, the
observations that the single spaceborne spectrum averages together.

In [ ]:
# --- Gather every AVIRIS-5 pixel that falls inside one EMIT pixel -------------------------------
# This is the basis of the two-scale comparison: a single ~60 m EMIT pixel covers the same ground as
# many ~10.4 m AVIRIS pixels. The gather function below returns every AVIRIS pixel whose centre
# lands inside the EMIT pixel footprint centred on a chosen (lat, lon), so §7.4 and §7.5 can show
# the spread of AVIRIS observations against the single EMIT spectrum.

# EMIT ortho-grid step (degrees) -> the pixel is one grid step wide in lon and lat. We halve it to
# get the footprint half-extent around a pixel centre.
_emit_dlon = float(np.median(np.abs(np.diff(ds_rfl["longitude"].values))))
_emit_dlat = float(np.median(np.abs(np.diff(ds_rfl["latitude"].values))))

# Report the footprint in metres, which shows how many AVIRIS pixels to expect inside it. One degree
# of latitude is ~111_132 m, one degree of longitude ~111_320 m scaled by cos(latitude); the EMIT
# ortho pixel is therefore square in degrees but wider north-south than east-west in metres.
_mlat = 111_132.0
_mlon = 111_320.0 * np.cos(np.radians(float(np.mean(ds_rfl["latitude"].values))))
_emit_m_ns, _emit_m_ew = _emit_dlat * _mlat, _emit_dlon * _mlon
print(f"EMIT pixel footprint: {_emit_m_ns:.1f} m (N-S) x {_emit_m_ew:.1f} m (E-W)")
print(f"AVIRIS-5 GSD: {AV_GSD:.1f} m  ->  expect ~{round((_emit_m_ns/AV_GSD)*(_emit_m_ew/AV_GSD))} "
      f"AVIRIS pixels per EMIT pixel")

# Per-site cache. Sections 7.2, 7.4 and 7.5 all ask for the same sites' pixels, and a
# full-spectrum read is the expensive kind (§6.4), so each site is read exactly once.
_AV_SITE_BLOCKS = {}


def aviris_pixels_in_emit_footprint(lat, lon):
    """All valid AVIRIS-5 spectra whose pixel centres fall inside the EMIT pixel at (lat, lon).

    Returns (wl_um, R) where R is a (n_pixels, n_bands) array of reflectance with atmospheric
    windows and fill (<=0) set to NaN.

    Cached per (lat, lon), because this is a full-spectrum read of a small block, the expensive
    kind described in §6.4, and every section below asks for the same handful of sites."""
    key = (round(float(lat), 6), round(float(lon), 6))
    if key in _AV_SITE_BLOCKS:
        return av_wl_um, _AV_SITE_BLOCKS[key]
    # EMIT pixel footprint corners (lat/lon), then project to the AVIRIS map CRS (easting/northing).
    lat0, lat1 = lat - _emit_dlat / 2, lat + _emit_dlat / 2
    lon0, lon1 = lon - _emit_dlon / 2, lon + _emit_dlon / 2
    (x0, y0), (x1, y1) = _av_proj_sel(lon0, lat0), _av_proj_sel(lon1, lat1)
    rows, cols = av_index_box(min(x0, x1), max(x0, x1), min(y0, y1), max(y0, y1))
    if rows[1] <= rows[0] or cols[1] <= cols[0]:
        _AV_SITE_BLOCKS[key] = np.empty((0, av_wl_um.size))
        return av_wl_um, _AV_SITE_BLOCKS[key]
    # A single windowed read of all 424 bands over this block: about 6 rows by 5 columns of AVIRIS
    # pixels around one EMIT centre, returned as (band, northing, easting). The number of pixels
    # actually inside the EMIT footprint is reported in section 7.2.
    blk = av_read(rows=rows, cols=cols)
    # Flatten the two spatial axes and transpose to (n_pixels, n_bands). The transpose returns a
    # non-contiguous view, and the next line writes into the result, so take an explicit copy.
    arr = blk.reshape(blk.shape[0], -1).T.copy()  # -> (n_pixels, n_bands); fill is already NaN
    arr[:, _AV_BAD] = np.nan                      # atmospheric windows / spectral tails (see §6.4)
    # Keep only spectra with at least half their bands finite. Pixels that fall on fill (outside the
    # flight line) or are heavily masked would otherwise contribute a flat, meaningless "spectrum".
    keep = np.isfinite(arr).mean(axis=1) >= 0.5
    _AV_SITE_BLOCKS[key] = arr[keep]
    return av_wl_um, _AV_SITE_BLOCKS[key]

# Quick sanity check: how many AVIRIS pixels actually fall in each site's EMIT footprint?
_t0 = time.time()
for lbl, pt, _mk, _cl in SITES:
    _, R = aviris_pixels_in_emit_footprint(*pt)
    print(f"{lbl:22s}: {R.shape[0]} valid AVIRIS pixels inside the EMIT footprint")
print(f"({len(SITES)} full-spectrum site reads in {time.time() - _t0:.1f} s, and they are now "
      f"cached, so every later section reuses them for free)")

<a id="section-7-3"></a>

### 7.3 Where the sites sit, in true color at both scales

Two true-color panels of the same ground: EMIT at ~60 m and AVIRIS-5 at ~10 m, with the sites
marked in both. The AVIRIS panel resolves what lies inside each EMIT pixel: outcrop, soil, shadow,
sparse vegetation, a road cut. That heterogeneity is the subject of §7.

**Figure 5. Comparison sites at both scales:** true-color EMIT and AVIRIS-5 over the same ground, with the site markers used throughout §7.

In [ ]:
def _stretch(rgb, lo=2, hi=98):
    '''Per-band percentile contrast stretch to [0,1], NaN-safe.

    Raw reflectance values are bunched into a narrow range, so a linear map from [min,max] looks flat.
    Mapping the 2nd and 98th percentiles to [0,1] and clipping instead spreads the mid-tones and
    ignores a few extreme outliers, the standard "percent-clip" stretch used for display RGBs.'''
    out = np.empty_like(rgb, dtype="float64")
    for b in range(rgb.shape[-1]):                       # stretch each of R, G, B independently
        band = rgb[..., b]
        finite = band[np.isfinite(band)]                 # ignore NaN (no-data / masked) pixels
        if finite.size == 0:
            out[..., b] = 0.0; continue                  # all-NaN band -> render as black
        p_lo, p_hi = np.percentile(finite, [lo, hi])     # the 2nd and 98th percentile values
        out[..., b] = np.clip((band - p_lo) / (p_hi - p_lo + 1e-9), 0, 1)
    return np.nan_to_num(out)                            # NaN -> 0 so imshow draws gaps as black

# --- EMIT RGB (dims: lat, lon, wavelengths) --------------------------------------------------------
# EMIT is small enough to slice all at once: pick the 3 visible bands and stretch to a display RGB.
emit_rgb_idx = [_nearest_band(wl_emit_um, t) for t in RGB_UM]
emit_rgb = _stretch(ds_rfl["reflectance"].values[:, :, emit_rgb_idx])
emit_lon = ds_rfl["longitude"].values
emit_lat = ds_rfl["latitude"].values

# --- AVIRIS-5 RGB ---------------------------------------------------------------------------------
# Section 6.5 read seven AVIRIS-5 bands over the ROI, the three visible ones plus four validity
# probes, onto the `av_east` / `av_north` grid and cached them as `av_rgb_raw`, so only the
# display stretch remains here.
# `_stretch()` ends with np.nan_to_num, which renders no-data as black. Black is also a plausible
# dark surface, so validity cannot be recovered from the stretched image afterwards, which is why
# §6.5 records it separately as `av_valid`.
av_rgb = _stretch(av_rgb_raw)

# The comparison sites used for the §7.4 and §7.5 multi-scale plots. These mark where the spectra
# are extracted from; they are EMIT-derived white-mica detections, not independent ground truth.
# The markers and colors are the ones assigned in §7.1, so this map serves as their frame of
# reference.
SAMPLE_SITES = SITES        # the three comparison sites selected in §7.1

# Transformer to project the sites' (lon, lat) into the AVIRIS-5 UTM CRS for the right-hand panel.
_to_av = Transformer.from_crs("EPSG:4326", av_crs_wkt, always_xy=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# EMIT panel (lon/lat axes); overlay the ROI outline and the three comparison sites.
# The EMIT panel is plotted in degrees, where one degree of longitude is shorter on the ground than
# one degree of latitude, so aspect="auto" is used; the AVIRIS panel below is in metres (UTM) and
# uses aspect="equal" to keep the pixels square.
axes[0].imshow(emit_rgb, origin="upper",
               extent=[emit_lon.min(), emit_lon.max(), emit_lat.min(), emit_lat.max()],
               aspect="auto")
roi_gdf.boundary.plot(ax=axes[0], color="red", linewidth=1.5)
for label, pt, mk, cl in SAMPLE_SITES:
    if pt:                                               # hollow marker so the imagery shows through
        axes[0].plot(pt[1], pt[0], mk, mfc="none", mec=cl, mew=2.2, ms=13, label=label)
axes[0].legend(loc="upper right", fontsize=8, framealpha=0.85)
axes[0].set_title(f"EMIT L2A RGB (~60 m)\n{sel_name}")
axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")

# AVIRIS-5 panel (easting/northing in native UTM); origin="upper" because northing is DESCENDING
# (row 0 = north), so the array's first row belongs at the top of the extent.
axes[1].imshow(av_rgb, origin="upper",
               extent=[av_east.min(), av_east.max(), av_north.min(), av_north.max()],
               aspect="equal")
for label, pt, mk, cl in SAMPLE_SITES:
    if pt:
        x, y = _to_av.transform(pt[1], pt[0])            # (lon, lat) -> (easting, northing)
        axes[1].plot(x, y, mk, mfc="none", mec=cl, mew=2.2, ms=13, label=label)
axes[1].legend(loc="upper right", fontsize=8, framealpha=0.85)
axes[1].set_title(f"AVIRIS-5 L2A RGB ({AV_GSD:.1f} m GSD)\n{aviris_line_name}")
axes[1].set_xlabel("Easting (m)"); axes[1].set_ylabel("Northing (m)")
axes[1].ticklabel_format(style="plain", useOffset=False)   # show full UTM metres, not 1e6 offset
axes[1].tick_params(axis="x", labelrotation=30)

fig.suptitle("Spectra-extraction sites at both scales, for inspection of cloud, "
             "shadow and vegetation before interpretation", fontweight="bold")
plt.tight_layout(); plt.savefig(FIG_DIR / "02_emit_aviris_rgb.png", dpi=130); plt.show()

<a id="section-7-4"></a>

### 7.4 Raw reflectance at both scales

Per site: every ~10 m AVIRIS pixel inside the ~60 m EMIT footprint (thin grey, the spread), their
mean (black, what a spatial average of the airborne data looks like), and the single EMIT spectrum
(colored, what the spaceborne sensor reports for that ground).

Expect the overall brightness to differ. The acquisitions are on different dates with different
illumination and different atmospheric corrections, and AVIRIS flies below most of the atmosphere.
Continuum removal in §7.5 removes that offset, so nothing should be read from the absolute
reflectance levels here.

**Figure 6. Raw reflectance:** EMIT and AVIRIS-5 spectra at each site, before continuum removal.

In [ ]:
# --- Raw reflectance, per site, at both scales -------------------------------------------------
def aviris_spectrum(lat, lon):
    """Single nearest-pixel AVIRIS-5 spectrum at (lat, lon): returns (wl_um, reflectance).

    A one-pixel, all-424-band read, the expensive shape described in §6.4, so it is used only
    where a single spectrum is what is needed. Anything requiring a site's whole neighbourhood
    goes through the cached aviris_pixels_in_emit_footprint() instead."""
    iy, ix = av_rowcol(lat, lon)
    if not (0 <= int(iy) < AV_NY and 0 <= int(ix) < AV_NX):
        return av_wl_um, np.full(av_wl_um.size, np.nan)     # site outside the ROI window
    r = av_read(rows=(int(iy), int(iy) + 1), cols=(int(ix), int(ix) + 1)).reshape(-1)
    return av_wl_um, su.bad_band_nan(av_wl_um, r)   # blank deep atmospheric-absorption windows


fig, axes = plt.subplots(1, len(SITES), figsize=(6.0 * len(SITES), 5), sharey=True, squeeze=False)
for ax, (label, pt, _mk, color) in zip(axes[0], SITES):
    wa, R = aviris_pixels_in_emit_footprint(*pt)
    for i in range(R.shape[0]):
        ax.plot(wa, R[i], color="0.6", lw=0.4, alpha=0.5,
                label=f"AVIRIS-5 pixels (n={R.shape[0]})" if i == 0 else None)
    if R.shape[0]:
        ax.plot(wa, np.nanmean(R, axis=0), color="k", lw=1.6, label="AVIRIS-5 mean")
    we, re = emit_spectrum(*pt)
    ax.plot(we, re, color=color, lw=2.2, label="EMIT (~60 m, 1 pixel)")
    ax.axvspan(*ALOH_WINDOW, color="gold", alpha=0.15)
    ax.set_title(f"{label}\n(AVIRIS {AV_GSD:.1f} m vs EMIT ~60 m)")
    ax.set_xlabel("Wavelength (µm)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0][0].set_ylabel("Reflectance")
fig.suptitle("Raw reflectance at matched sites - the many AVIRIS pixels inside one EMIT pixel",
             fontweight="bold")
plt.tight_layout(); plt.savefig(FIG_DIR / "02_multiscale_spectra.png", dpi=130,
                                bbox_inches="tight"); plt.show()

<a id="section-7-5"></a>

### 7.5 Continuum-removed, on a shared baseline

This is the comparison that supports quantitative interpretation. Every curve is continuum-removed
over the same
2.10–2.26 µm window, so all of them start and end at 1.0 and the dip is the true band depth. On a
shared baseline you can read position (where each curve bottoms out) and depth directly
against each other, across sensors and across sites.

What to look for:

- Do the EMIT and AVIRIS-mean curves bottom out at the same wavelength? A systematic offset would
  point at a calibration or continuum-window issue, not geology.
- Is the EMIT feature shallower than the AVIRIS mean? It usually is, because sub-pixel mixing
  with spectrally-flat material dilutes depth. That is why depth is a poor mineral-abundance proxy
  across sensors while position is comparatively robust.
- Does the spread of the thin grey curves differ between sites? A site where all ~26 airborne
  spectra agree is a homogeneous EMIT pixel; a site where they scatter is a mixed one, and its
  single EMIT number should be read with much more caution.

**Figure 7. Continuum-removed spectra:** all sites on a shared 1.0 baseline, so the Al-OH feature positions can be read against each other.

In [ ]:
# --- Continuum-removed spectra, all sites, shared 1.0 baseline ---------------------------------
fig, ax = plt.subplots(figsize=(10, 6.5))
# `rows` accumulates one (sensor, site, metrics dict, AVIRIS position spread, n pixels) tuple per
# measured curve; the summary table below unpacks it in that order, so the layout matters.
rows = []


def _cr_line(w_um, r, color, ls, lw, alpha=1.0, label=None):
    cw, cr = su.continuum_removed(w_um, r, wl_range=ALOH_WINDOW)
    ax.plot(cw, cr, color=color, lw=lw, ls=ls, alpha=alpha, label=label)


for label, pt, _mk, color in SITES:
    # Individual AVIRIS pixels inside the EMIT footprint (thin, faint) + their feature positions.
    wa, R = aviris_pixels_in_emit_footprint(*pt)
    av_pos, av_depth = [], []
    for i in range(R.shape[0]):
        _cr_line(wa, R[i], color, "-", 0.4, alpha=0.30,
                 label=f"{label}: AVIRIS pixels (n={R.shape[0]})" if i == 0 else None)
        m = su.aloh_feature_metrics(wa, R[i], wl_range=ALOH_WINDOW)
        if np.isfinite(m["min_wavelength_um"]):
            av_pos.append(m["min_wavelength_um"])
            av_depth.append(m["band_depth"])

    # AVIRIS mean (dashed, bold).
    if R.shape[0]:
        rmean = np.nanmean(R, axis=0)
        _cr_line(wa, rmean, color, "--", 1.6, label=f"{label}: AVIRIS mean")
        met_av = su.aloh_feature_metrics(wa, rmean, wl_range=ALOH_WINDOW)
        rows.append(("AVIRIS-5 mean", label, met_av,
                     np.nanstd(av_pos) if av_pos else np.nan, R.shape[0]))

    # EMIT single pixel (solid, bold).
    we, re = emit_spectrum(*pt)
    _cr_line(we, re, color, "-", 2.2, label=f"{label}: EMIT (~60 m)")
    met_e = su.aloh_feature_metrics(we, re, wl_range=ALOH_WINDOW)
    if np.isfinite(met_e["min_wavelength_um"]):
        ax.axvline(met_e["min_wavelength_um"], color=color, ls=":", lw=1, alpha=0.7)
    rows.append(("EMIT", label, met_e, np.nan, 1))

ax.axhline(1.0, color="k", lw=0.8, alpha=0.4)        # the shared continuum reference
# Reference positions along the muscovite-phengite solid solution, annotated just above the
# continuum baseline (y = 1.01) so the measured minima can be read against them directly.
for x, lbl in [(2.190, "Al-rich mica"), (2.200, "muscovite"), (2.210, "phengitic")]:
    ax.text(x, 1.01, lbl, rotation=90, fontsize=7, color="0.35", va="bottom", ha="center")
ax.set_xlabel("Wavelength (µm)"); ax.set_ylabel("Continuum-removed reflectance")
ax.set_title("Al-OH absorption feature, continuum removed (shared 1.0 baseline)")
ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR / "02_continuum_removed.png", dpi=130,
                                bbox_inches="tight"); plt.show()

The table below reports, for each sensor and site, the Al-OH position (the composition / pH
proxy), the depth, and the FWHM. For the AVIRIS means it also reports how much the position
varies from pixel to pixel within the single EMIT footprint, and how many pixels the mean is built
from, which measures the heterogeneity the spaceborne pixel conceals.

In [ ]:
# --- Summary table: Al-OH feature metrics per (sensor, site) -----------------------------------
# Unpack the tuples assembled in §7.5 into a tidy table: feature positions in µm, and the
# within-footprint spread converted to nanometres, which is the scale the offsets are judged on.
metrics_df = pd.DataFrame(
    [{"sensor": s, "site": lbl,
      "Al-OH position (µm)": round(m["min_wavelength_um"], 4),
      "band depth": round(m["band_depth"], 3),
      "FWHM (µm)": round(m["fwhm_um"], 4),
      "AVIRIS pos. spread (nm)": (round(pos_std * 1000.0, 1) if np.isfinite(pos_std) else np.nan),
      "n pixels": n}
     for (s, lbl, m, pos_std, n) in rows]
).sort_values(["site", "sensor"]).reset_index(drop=True)
metrics_df

<a id="section-7-6"></a>

### 7.6 The same measurement, mapped at both scales

Numbers at three sites establish whether the comparison holds there. A map shows whether the
spatial pattern survives the change in resolution, which is the property a position map must have
before it can be used to vector toward a fluid center.

We compute the Al-OH center on a small window covering the white-mica sites, at both resolutions, and
show them side by side. `WINDOW_DEG` sets the window size and `STRIDE` can subsample the airborne
grid, though at `STRIDE = 1` (full 10 m resolution) this cell still runs in a few seconds.
`WINDOW_DEG` is a half-size, so the window spans twice that value in each direction.

This is a wide-area, few-band read: roughly 30 of AVIRIS's 424 bands fall inside the Al-OH
window, which §6.4 showed is the cheap direction to cut a tiled spectral cube. The feature fit is
also vectorized: `su.aloh_metrics_cube` applies the same endpoint continuum and the same three-point
parabolic refinement as the scalar `su.aloh_feature_metrics` of §5.2, but as numpy operations over
every pixel at once. The per-pixel Python loop doing the identical arithmetic takes minutes on the
same array. The two implementations are cross-checked against each other on a random sample of
pixels below.

**Figure 8. Al-OH center at two scales:** the same ground mapped by EMIT and AVIRIS-5 on an identical color stretch. EMIT is on a latitude/longitude grid and AVIRIS-5 on its native UTM grid, so these panels stay in matplotlib.

In [ ]:
# --- Al-OH center map at both scales over a common window --------------------------------------
STRIDE = 1            # subsample the AVIRIS grid (1 = full 10 m resolution)
WINDOW_DEG = 0.020    # window half-size in degrees (~2.2 km); covers the white-mica sites
AV_DEPTH_MIN = 0.02   # only map AVIRIS pixels with a real feature

# Center the window on the comparison sites, every site in this notebook is a white mica, and
# fall back to the centre of the EMIT grid if §7.1 found no valid site.
_wm_sites = [(lbl, pt) for lbl, pt, _m, _c in SITES]
_ctr_lat = float(np.mean([pt[0] for _l, pt in _wm_sites])) if _wm_sites else float(np.mean(lat))
_ctr_lon = float(np.mean([pt[1] for _l, pt in _wm_sites])) if _wm_sites else float(np.mean(lon))
w_lat0, w_lat1 = _ctr_lat - WINDOW_DEG, _ctr_lat + WINDOW_DEG
w_lon0, w_lon1 = _ctr_lon - WINDOW_DEG, _ctr_lon + WINDOW_DEG
print(f"common window: lat {w_lat0:.4f} to {w_lat1:.4f}, lon {w_lon0:.4f} to {w_lon1:.4f}")

# ---- AVIRIS side: one wide, few-band read, then a single vectorized feature fit ----
_av_win_idx = np.where((av_wl_um >= ALOH_WINDOW[0]) & (av_wl_um <= ALOH_WINDOW[1]))[0]
_w_av = av_wl_um[_av_win_idx]
(_x0, _y0) = _av_proj_sel(w_lon0, w_lat0)
(_x1, _y1) = _av_proj_sel(w_lon1, w_lat1)
_rows, _cols = av_index_box(min(_x0, _x1), max(_x0, _x1), min(_y0, _y1), max(_y0, _y1))
print(f"AVIRIS window: {_rows[1] - _rows[0]} x {_cols[1] - _cols[0]} pixels at "
      f"{AV_GSD * STRIDE:.1f} m effective, {_av_win_idx.size} of {av_wl_um.size} bands "
      f"-> reading ...")

_t0 = time.time()
# Bands first (as GDAL stores them), then move the spectral axis last for the cube fit.
# GDAL numbers raster bands from 1, so the zero-based numpy indices are shifted by one. The STRIDE
# slices subsample the two spatial axes only, never the spectral axis.
_cube = av_read(bands=_av_win_idx + 1, rows=_rows, cols=_cols)[:, ::STRIDE, ::STRIDE]
_cube = np.moveaxis(_cube, 0, -1)              # -> (northing, easting, band); fill already NaN
print(f"  read {_cube.nbytes / 1e6:.0f} MB in {time.time() - _t0:.1f} s -> fitting ...")

_t0 = time.time()
# min_valid_frac=0.8 requires 80% of the in-window bands to be finite before a pixel is fitted,
# which rejects fill and heavily masked spectra without discarding pixels missing a band or two.
av_pos_map, _av_depth_map = su.aloh_metrics_cube(_w_av, _cube, wl_range=ALOH_WINDOW,
                                                 min_valid_frac=0.8)
av_pos_map = np.where(_av_depth_map >= AV_DEPTH_MIN, av_pos_map, np.nan)   # real feature only
print(f"  fit {av_pos_map.size:,} pixels in {time.time() - _t0:.2f} s")
print(f"  measured {int(np.isfinite(av_pos_map).sum()):,} AVIRIS centers "
      f"({100 * np.isfinite(av_pos_map).mean():.0f}% of the window)")

# Cross-check the vectorized fit against the scalar reference implementation of §5.3 on a random
# sample of measured pixels. They should agree to well under a nanometre.
_iy_s, _ix_s = np.where(np.isfinite(av_pos_map))
if _iy_s.size:
    # Cross-check the vectorized fit against the scalar reference implementation on up to 25
    # pixels. The generator is seeded so the sample, and therefore the printed discrepancy, is
    # reproducible from run to run.
    _pick = np.random.default_rng(0).choice(_iy_s.size, size=min(25, _iy_s.size), replace=False)
    _dmax = max(abs(av_pos_map[_iy_s[k], _ix_s[k]]
                    - su.aloh_feature_metrics(_w_av, _cube[_iy_s[k], _ix_s[k]],
                                              wl_range=ALOH_WINDOW)["min_wavelength_um"])
                for k in _pick)
    print(f"  vectorized vs per-pixel reference on {len(_pick)} random pixels: "
          f"max difference {1000 * _dmax:.5f} nm")

# ---- EMIT side: just crop the section 5.3 map to the same window ----
_ey = (lat >= w_lat0) & (lat <= w_lat1)
_ex = (lon >= w_lon0) & (lon <= w_lon1)
# np.ix_ turns the two 1-D boolean selectors into an open mesh, cropping rows AND columns of the
# section 5.3 map in one step (wm_map[_ey, _ex] would instead pair the selectors element-wise).
emit_pos_win = wm_map[np.ix_(_ey, _ex)]
print(f"EMIT window: {int(_ey.sum())} x {int(_ex.sum())} pixels, "
      f"{int(np.isfinite(emit_pos_win).sum()):,} measured centers")

# ---- Side-by-side ----
# Both panels use the shared VMIN / VMAX stretch defined with the measurement in §5.3, so the two
# sensors here and Figure 9 in §7.7 all sit on one colour scale.
fig, (ax_e, ax_a) = plt.subplots(1, 2, figsize=(15, 7))

pcm_e = ax_e.pcolormesh(lon[_ex], lat[_ey], emit_pos_win, cmap="RdYlBu_r",
                        vmin=VMIN, vmax=VMAX, shading="auto")
ax_e.set_title(f"EMIT (~60 m)\n{int(np.isfinite(emit_pos_win).sum()):,} white-mica pixels")
ax_e.set_xlabel("Longitude"); ax_e.set_ylabel("Latitude")
fig.colorbar(pcm_e, ax=ax_e, shrink=0.7, label="Al-OH center (µm)")

_av_e_win = av_east[_cols[0]:_cols[1]][::STRIDE]     # the ROI-grid coordinates this window covers
_av_n_win = av_north[_rows[0]:_rows[1]][::STRIDE]
_e_ax = [float(_av_e_win.min()), float(_av_e_win.max()),
         float(_av_n_win.min()), float(_av_n_win.max())]
# Row 0 of av_pos_map is the NORTHERNmost row (northing descends down the grid), so it belongs at
# the top of the extent -> origin="upper".
im_a = ax_a.imshow(av_pos_map, origin="upper", extent=_e_ax,
                   cmap="RdYlBu_r", vmin=VMIN, vmax=VMAX, aspect="equal")
ax_a.set_title(f"AVIRIS-5 ({AV_GSD * STRIDE:.1f} m effective)\n"
               f"{int(np.isfinite(av_pos_map).sum()):,} pixels with a feature")
ax_a.set_xlabel("Easting (m)"); ax_a.set_ylabel("Northing (m)")
ax_a.ticklabel_format(style="plain", useOffset=False)
ax_a.tick_params(axis="x", labelrotation=30)
fig.colorbar(im_a, ax=ax_a, shrink=0.7, label="Al-OH center (µm)")

for lbl, pt, mk, cl in SITES:
    if w_lat0 <= pt[0] <= w_lat1 and w_lon0 <= pt[1] <= w_lon1:
        ax_e.plot(pt[1], pt[0], mk, mfc="none", mec="k", mew=2.0, ms=13, label=lbl)
        _x, _y = _av_proj_sel(pt[1], pt[0])
        ax_a.plot(_x, _y, mk, mfc="none", mec="k", mew=2.0, ms=13, label=lbl)
if ax_e.get_legend_handles_labels()[0]:
    ax_e.legend(loc="upper right", fontsize=8, framealpha=0.85)

fig.suptitle("Al-OH absorption center over the same ground at two scales "
             "(identical color stretch)", fontweight="bold")
plt.tight_layout(); plt.savefig(FIG_DIR / "02_center_map_two_scales.png", dpi=130,
                                bbox_inches="tight"); plt.show()

_ev = emit_pos_win[np.isfinite(emit_pos_win)]
_av = av_pos_map[np.isfinite(av_pos_map)]
if _ev.size and _av.size:
    print(f"\nwindow medians: EMIT {np.median(_ev):.4f} µm  vs  AVIRIS {np.median(_av):.4f} µm "
          f"({1000 * (np.median(_ev) - np.median(_av)):+.1f} nm)")
    print(f"window spread (sd): EMIT {1000 * np.std(_ev):.1f} nm  vs  "
          f"AVIRIS {1000 * np.std(_av):.1f} nm")
    print("\nThe AVIRIS panel is not simply a sharper copy of the EMIT panel. It maps many more")
    print("pixels, because features diluted below detection in a 60 m mixture are recoverable at")
    print("10 m - and its wider spread is real compositional variation that EMIT averages away.")

<a id="section-7-7"></a>

### 7.7 The full-scene picture: detection next to composition

The white-mica detection map from §4.5 is placed beside the full-AOI absorption-center map from
§5.3. Same extent, same basemap, same site markers.

The left panel is what the classifier provides: where white mica is, and how strongly the feature is
expressed. The right panel adds the information the left one discards: within that same alteration,
where the fluid was more Al-rich (shorter wavelength, higher pH) and where more Fe–Mg-substituted.
The two panels can look almost identical in extent while carrying completely different information.

**Figure 9. Detection next to composition:** the same scene read two ways. The left panel is
white-mica detection strength from §4.5, the right panel is the Al-OH absorption center from §5.3 on
the same 2.190 to 2.215 µm stretch used in §7.6. The linked axes pan and zoom together, so a
feature can be followed between the two readings.

In [ ]:
# --- White-mica detection (left) vs Al-OH position / fluid-composition proxy (right) -------------
# One GeoDataFrame per site so each gets its own color and legend entry, as in the matplotlib
# panels above.
# Bokeh names its marker shapes, matplotlib abbreviates them; this keeps each site's symbol the
# same here as in the matplotlib panels of §7.3 and §7.6.
HV_MARKER = {"s": "square", "o": "circle", "^": "triangle"}
site_layers = hv.Overlay([
    gpd.GeoDataFrame(geometry=[Point(pt[1], pt[0])], crs="EPSG:4326").hvplot.points(
        geo=True, color=cl, size=130, marker=HV_MARKER[mk], line_color="white",
        line_width=1.5, label=lbl, hover=False)
    for lbl, pt, mk, cl in SITES
])

MAP_OPTS = dict(x="longitude", y="latitude", geo=True, tiles="EsriImagery",
                alpha=0.85, frame_width=520, frame_height=560)

detection_panel = wm["band_depth"].hvplot.image(
    **MAP_OPTS, cmap="Blues", clim=(BAND_DEPTH_MIN, BD_HI),
    clabel="white-mica Al-OH band depth",
    title="White-mica detection strength (§4.5)",
).opts("Image", xrotation=45)

# The same 2.190-2.215 µm stretch and RdYlBu_r colormap as the two-scale maps above, so colors are
# directly comparable between the two figures. Short wavelength = Al-rich, higher pH; long
# wavelength = Fe-Mg substituted, lower pH.
center_panel = wm["center"].hvplot.image(
    **MAP_OPTS, cmap="RdYlBu_r", clim=(VMIN, VMAX),
    clabel="Al-OH center (µm):  <- Al-rich, higher pH   |   Fe-Mg, lower pH ->",
    title="White-mica Al-OH center (§5.3)",
).opts("Image", xrotation=45)

composition_figure = ((detection_panel * roi_outline * site_layers)
                      + (center_panel * roi_outline * site_layers)).cols(2)
composition_figure = composition_figure.opts(hv.opts.Layout(
    title="Same scene, two readings: where white mica is, and what composition it records"))
composition_figure

<a id="section-8"></a>

## 8. Interpretation and next steps

### Summary of the workflow

This notebook identified white micas over the White Mica AOI with EMIT L2B_MIN, returned to the
L2A reflectance to measure the Al-OH absorption center at every white-mica pixel, and tested
that measurement against coincident AVIRIS-5 airborne reflectance at ~6× finer resolution, first
at three sites chosen to span the compositional range, then as a map.

### Three concepts to carry forward

A classified mineral map is a lossy summary of the measurement. The L2B_MIN library already
distinguishes low-, medium- and high-Al muscovite; collapsing them into "White mica" discards the
compositional axis that makes white mica useful as a fluid-chemistry vector (§4.3). Going back to
the reflectance and measuring the feature position recovers it as a continuous quantity, and the
detection map and the position map cover nearly the same ground while answering entirely different
questions (§7.7).

Feature position transfers across sensors and scales; band depth does not. Continuum removal
makes feature position comparable between two instruments flown on different days at different
altitudes with different atmospheric corrections. Band depth does not survive that trip, because
sub-pixel mixing with spectrally-flat material dilutes it, which is why the EMIT features in §7.5
are shallower than the AVIRIS means and why depth is a poor cross-sensor abundance proxy.

A 60 m spectral measurement is a spatial mixture, and mixing does not commute with the
non-linear feature fit.
Reflectance mixes area-weighted, but absorption-center position is a non-linear function of the
spectrum, so the EMIT center need not sit at the mean of the ~26 AVIRIS centers beneath it. The §7.5
summary table reports the within-footprint spread of the airborne positions alongside each EMIT
value; judge it against EMIT's ~7.4 nm band spacing and the ~25 nm width of the whole white-mica
range. Where the spread is a few nm and the EMIT center sits inside it, the position map can be read
as mineral chemistry. Where it does not, the map is still a useful pattern, but the numbers on the
colorbar are not defensible as composition.

### Reading the result geologically

The absorption-center map (§7.7, right panel) is a fluid-chemistry vector map, not a mineral
map. Shorter wavelengths mark Al-rich muscovitic mica, generally grown from a more acidic-to-neutral
fluid closer to the upflow zone; longer wavelengths mark Fe–Mg-substituted phengitic mica, typically
more distal, cooler, or in more mafic host rock. Traced across the AOI, the gradient sketches the pH
and thermal architecture of the alteration, which is what points at the underlying intrusive centre
and its lithophile critical-metal suite (Be, Li, Mo, Sn, W, REE, U, F) in these Blawn Formation
A-type rhyolites.

Two caveats apply to any such reading. Host-rock composition also affects mica chemistry, so a
long-wavelength anomaly over a mafic unit is not necessarily a distal fluid, and the pattern should
be checked against a geologic map of the host units. The EMIT scene and the AVIRIS flight are also on
different dates, so a discrepancy at a site can reflect a change on the ground (moisture,
vegetation, disturbance) rather than a scale effect.

### Where to go next

- Tighten the composition proxy by pairing position with the 2.35 µm Mg-OH feature, which
  responds to the same substitution and breaks some of the degeneracy in the 2.2 µm position alone.
- Bring in the temporal dimension from notebook 1: aggregate the white-mica detection across
  many dates to establish where white mica is persistently detected, then measure the center only
  there. Persistence and composition answer different questions and are stronger together.
- Validate against ground truth. Notebook 1's approach, comparing a remote product against
  independent surface geochemistry, applies directly here if you have mica separates with
  microprobe or XRD compositions.
- Transfer the workflow. Nothing in §4–§7 is specific to this district; substituting the ROI
  yields a white-mica composition workflow for any lithocap with an AVIRIS-5 or EMIT overpass.

### Data citations

- EMIT L2A Reflectance and Uncertainty, V001. LP DAAC. doi:10.5067/EMIT/EMITL2ARFL.001
- EMIT L2B Estimated Mineral Identification and Band Depth, V001. LP DAAC.
  doi:10.5067/EMIT/EMITL2BMIN.001
- AVIRIS-5 L2A Orthorectified Surface Reflectance. ORNL DAAC (CMR `C4184090548-ORNL_CLOUD`).
- Clark, R.N. & Roush, T.L. (1984) Reflectance spectroscopy: quantitative analysis techniques.
  JGR 89, 6329–6340.
- Meyer, J.M., Kokaly, R.F. & Holley, E. (2022) Hyperspectral remote sensing of white mica: a review
  of imaging and point-based spectrometer studies for mineral resources, with spectrometer design
  considerations. Remote Sensing of Environment 275, 113000. doi:10.1016/j.rse.2022.113000
- Graham, G.E., Kokaly, R.F., Kelley, K.D., Hoefen, T.M., Johnson, M.R. & Hubbard, B.E. (2018)
  Application of imaging spectroscopy for mineral exploration in Alaska: a study over porphyry Cu
  deposits in the eastern Alaska Range. Economic Geology 113, 489–510.
  doi:10.5382/econgeo.2018.4559
- Lindsey, D.A. & Osmonson, L.M. (1978) USGS OFR 78-114 (Blawn Mountain / Wah Wah alunite).
- Barkoff, D. (2022) PhD dissertation, UNLV. doi:10.34917/35777457
- Portela, B. et al. (2025) Ore Geology Reviews 182, 106673.

### Software and format references

- GDAL Virtual Format (VRT) raster driver, used in §6 to expose the AVIRIS-5 NetCDF reflectance
  variable as a windowed raster: https://gdal.org/en/stable/drivers/raster/vrt.html

Companion notebook: `01_alunite_temporal_aggregation.ipynb`, multi-date aggregation of EMIT
L2B_MIN over an alunite lithocap, tested against UGS surface geochemistry.

## Contact Info:

Email: LPDAAC@usgs.gov
Voice: +1-866-573-3222
Organization: Land Processes Distributed Active Archive Center (LP DAAC)¹
Website: <https://www.earthdata.nasa.gov/centers/lp-daac>

¹Work performed under USGS contract 140G0126D0001 for NASA contract NNG14HH33I.